In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Generating Table

This creates the tables of the accuracies and clustering scores for distinguishing simulated graph distributions. Note that you first have to run `sim_distributions.py`.

In [2]:
# getting the right parameters
model_params = {
        'ER': {'ER_probs': [0.75], 
                'ER_nps': None,
        },
        'RPG':{
            'RPG_probs': [[0.9, 0.1]], 
            'RPG_props': [1/3, 1/3], 
        },
        'SBM': {
            'SBM_probs': [0.8, 0.05], 
            'SBM_blocks': [[1/4, 1/4, 1/4]]
            },
        'RG': {
            'RG_probs': [0.25]
        },
    }
    
models = list(model_params.keys())
all_configs = {}
for model in models:
    model_config = tuple(model_params[model][param] for param in model_params[model].keys())
    all_configs[model] = model_config
configuration = tuple(all_configs[model] for model in models)

# read dataframe
res_df = pd.read_csv(f'./simulations/results/sim_distributions/res_df.csv', header=0, index_col=0)


# group methods
method_group = {}
for method in list(res_df.index):
   if 'toper' in method:
      method_group[method] = 'TopER'
   elif 'kernel' in method:
      method_group[method] = 'Kernel'
   elif 'embedding' in method or 'spectral_zoo' == method:
      method_group[method] = 'Embeddings'
   elif 'divcurves' in method:
      method_group[method] = 'Diversity Curves'
   elif 'div_function' in method or 'one_scale' in method:
      method_group[method] = 'Diversity Profile'
   elif 'persistent_homology' in method:
      method_group[method] = 'Persistent Homology'

      
res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
      'divcurves_shortest_path_distance': 'Shortest Path',
      'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
      'div_function_shortest_path_distance': 'Profile Shortest Path', 
      'graph_embedding_netLSD': 'NetLSD',
      'graph_embedding_graph2vec': 'Graph2Vec', 
      'graph_embedding_feathergraph': 'FEATHER',
      'kernel_shortest_path': 'Kernel Shortest Path', 
      'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
      'kernel_pyramid_match': 'Kernel Pyramid Match', 
      'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
      'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
      'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
      'kernel_weisfeiler_lehman': 'Kernel WL',
      'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
      'kernel_core_framework': 'Kernel Core Framework',
      'kernel_mds_core_framework': 'Kernel Core Framework MDS',
      'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
      'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
      'toper_degree_sub': 'TopER Degree', 
      'toper_popularity_sub': 'TopER Popularity', 
      'toper_forricci_sub': 'TopER Forricci',
      'toper_olricci_sub': 'TopER Olricci',
      'toper_closeness_sub': 'TopER Closeness', 
      'toper_all_sub': 'TopER All Filtrations',
      'spectral_zoo': 'Spectral Zoo', 
      'persistent_homology_degree': 'PH Degree',
      'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',
      'persistent_homology_forman_ricci_curvature': 'PH Fricci',
      'persistent_homology_laplacian': 'PH Laplacian', 
      'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
      'persistent_homology_alpha_complex': 'PH Alpha Complex',
}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)

# sort wrt to accuracy, in groups
df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)

scores = ['acc', 'silh', 'db', 'ch', 'ari_kmeans', 'ari_agglo']
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

res_copy = df.copy()

results_latex = {}

for score in scores:
   mean_col = f'{score}_mean'
   std_col = f'{score}_std'

   if mean_col in res_copy.columns and std_col in res_copy.columns:
      results_latex[score] = res_copy[mean_col].combine(
            res_copy[std_col],
            lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
      )

results_latex_df = pd.DataFrame.from_dict(results_latex)

# Bold the best mean for each score 
for score in scores:
   mean_col = f"{score}_mean"
   display_col =  score
   if mean_col in res_copy.columns:
      if not scores_asc[score]: 
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
      else:
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
      results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
            lambda x: f"\\textbf{{{x}}}"
      )
   else:
      print(f'{mean_col} not in dataframe?')

#final adjustments latex df 
results_latex_df.index.rename([None, 'Method'], inplace = True)
new_cols = {
   'acc': 'Accuracy', 
   'silh': 'Silhouette Score', 
   'db': 'Davies-Bouldin', 
   'ch': 'Calinski-Harabasz', 
   'ari_kmeans': 'ARI k-means', 
   'ari_agglo': 'ARI agglomerative'
}

results_latex_df = results_latex_df.rename(columns=new_cols)
results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

print(configuration)
print(results_latex_df.to_latex(multirow=False, escape=False))

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &            &                     Accuracy &             Silhouette Score &               Davies-Bouldin &               Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                 &                              &                              \\
\midrule
Diversity Curves & Shortest Path &  \textbf{0.904 $\pm $ 0.042} &           0.427 $\pm $ 0.034 &           0.748 $\pm $ 0.060 &           194.340 $\pm $ 34.081 &  \textbf{0.511 $\pm $ 0.081} &           0.311 $\pm $ 0.119 \\
                    & Shortest Path PCA &           0.888 $\pm $ 0.053 &  \textbf{0.436 $\pm $ 0.034} &  \textbf{0.732 $\pm $ 0.060} &  \textbf{197.355 $\pm $ 35.008} &  \textbf{0

## nrange50_100

In [25]:
# getting the right parameters
model_params = {
        'ER': {'ER_probs': [0.75], 
                'ER_nps': None,
        },
        'RPG':{
            'RPG_probs': [[0.9, 0.1]], 
            'RPG_props': [1/3, 1/3], 
        },
        'SBM': {
            'SBM_probs': [0.8, 0.05], 
            'SBM_blocks': [[1/4, 1/4, 1/4]]
            },
        'RG': {
            'RG_probs': [0.25]
        },
    }
    
models = list(model_params.keys())
all_configs = {}
for model in models:
    model_config = tuple(model_params[model][param] for param in model_params[model].keys())
    all_configs[model] = model_config
configuration = tuple(all_configs[model] for model in models)

# read dataframe
res_df = pd.read_csv(f'./simulations/results/sim_distributions/nrange50_100/res_df.csv', header=0, index_col=0)
#step compute cardinality: 5

# group methods
method_group = {}
for method in list(res_df.index):
   if 'toper' in method:
      method_group[method] = 'TopER'
   elif 'kernel' in method:
      method_group[method] = 'Kernel'
   elif 'embedding' in method or 'spectral_zoo' == method:
      method_group[method] = 'Embeddings'
   elif 'divcurves' in method:
      method_group[method] = 'Diversity Curves'
   elif 'div_function' in method or 'one_scale' in method:
      method_group[method] = 'Diversity Profile'
   elif 'persistent_homology' in method:
      method_group[method] = 'Persistent Homology'

      
res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
      'divcurves_shortest_path_distance': 'Shortest Path',
      'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
      'div_function_shortest_path_distance': 'Profile Shortest Path', 
      'graph_embedding_netLSD': 'NetLSD',
      'graph_embedding_graph2vec': 'Graph2Vec', 
      'graph_embedding_feathergraph': 'FEATHER',
      'kernel_shortest_path': 'Kernel Shortest Path', 
      'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
      'kernel_pyramid_match': 'Kernel Pyramid Match', 
      'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
      'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
      'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
      'kernel_weisfeiler_lehman': 'Kernel WL',
      'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
      'kernel_core_framework': 'Kernel Core Framework',
      'kernel_mds_core_framework': 'Kernel Core Framework MDS',
      'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
      'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
      'toper_degree_sub': 'TopER Degree', 
      'toper_popularity_sub': 'TopER Popularity', 
      'toper_forricci_sub': 'TopER Forricci',
      'toper_olricci_sub': 'TopER Olricci',
      'toper_closeness_sub': 'TopER Closeness', 
      'toper_all_sub': 'TopER All Filtrations',
      'spectral_zoo': 'Spectral Zoo', 
      'persistent_homology_degree': 'PH Degree',
      'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',
      'persistent_homology_forman_ricci_curvature': 'PH Fricci',
      'persistent_homology_laplacian': 'PH Laplacian', 
      'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
      'persistent_homology_alpha_complex': 'PH Alpha Complex',
}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)

# sort wrt to accuracy, in groups
df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)

scores = ['acc', 'silh', 'db', 'ch', 'ari_kmeans', 'ari_agglo']
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

res_copy = df.copy()

results_latex = {}

for score in scores:
   mean_col = f'{score}_mean'
   std_col = f'{score}_std'

   if mean_col in res_copy.columns and std_col in res_copy.columns:
      results_latex[score] = res_copy[mean_col].combine(
            res_copy[std_col],
            lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
      )

results_latex_df = pd.DataFrame.from_dict(results_latex)

# Bold the best mean for each score 
for score in scores:
   mean_col = f"{score}_mean"
   display_col =  score
   if mean_col in res_copy.columns:
      if not scores_asc[score]: 
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
      else:
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
      results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
            lambda x: f"\\textbf{{{x}}}"
      )
   else:
      print(f'{mean_col} not in dataframe?')

#final adjustments latex df 
results_latex_df.index.rename([None, 'Method'], inplace = True)
new_cols = {
   'acc': 'Accuracy', 
   'silh': 'Silhouette Score', 
   'db': 'Davies-Bouldin', 
   'ch': 'Calinski-Harabasz', 
   'ari_kmeans': 'ARI k-means', 
   'ari_agglo': 'ARI agglomerative'
}

results_latex_df = results_latex_df.rename(columns=new_cols)
results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

print(configuration)
print(results_latex_df.to_latex(multirow=False, escape=False))

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &                  &                     Accuracy &             Silhouette Score &               Davies-Bouldin &                 Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                   &                              &                              \\
\midrule
Diversity Curves & Shortest Path &  \textbf{1.000 $\pm $ 0.000} &           0.686 $\pm $ 0.009 &           0.417 $\pm $ 0.013 &           1292.182 $\pm $ 124.201 &           0.755 $\pm $ 0.138 &           0.545 $\pm $ 0.176 \\
                    & Shortest Path PCA &  \textbf{1.000 $\pm $ 0.000} &           0.695 $\pm $ 0.009 &           0.403 $\pm $ 0.013 &           1331.783 $\pm $ 131.40

In [26]:
df

silh_mean    db_mean  \
Group               Method                                                   
Diversity Curves    Shortest Path                      0.685895   0.416718   
                    Shortest Path PCA                  0.695339   0.402883   
Diversity Profile   Profile Shortest Path              0.249913   1.504802   
Embeddings          NetLSD                             0.564212   0.591856   
                    FEATHER                            0.361271   0.869947   
                    Spectral Zoo                       0.300955   0.824624   
                    Graph2Vec                          0.093721   6.406873   
Kernel              Kernel Shortest Path               0.907858        NaN   
                    Kernel Shortest Path MDS           0.847388   0.219886   
                    Kernel WL                          0.794931        NaN   
                    Kernel WL MDS                      0.826739   0.251361   
                    Kernel Core Framework MDS          0.644327   0.495460   
                    Kernel Core Framework              0.622316        NaN   
                    Kernel WL Optimal Assignment       0.032167        NaN   
                    Kernel WL Optimal Assignment MDS   0.095817   2.152891   
                    Kernel Pyramid Match              -0.004319        NaN   
                    Kernel Pyramid Match MDS          -0.032477  34.799833   
                    Kernel Neighbourhood Hash MDS     -0.037285  56.933585   
                    Kernel Neighbourhood Hash         -0.024274        NaN   
TopER               TopER All Filtrations              0.051297   1.963484   
                    TopER Closeness                    0.014251   1.836302   
                    TopER Degree                       0.038695   1.536480   
                    TopER Popularity                   0.005106   3.888972   
                    TopER Olricci                     -0.006826   1.734721   
                    TopER Forricci                     0.048295  16.726045   
Persistent Homology PH Vietoris Rips                   0.277375   1.603607   
                    PH Fricci                          0.127746   1.159742   
                    PH Olricci                         0.135910   1.001598   
                    PH Laplacian                      -0.028093   1.791270   
                    PH Degree                         -0.055423   2.011745   
                    PH Alpha Complex                   0.209755   3.091385   

                                                          ch_mean  silh_std  \
Group               Method                                                    
Diversity Curves    Shortest Path                     1292.182049  0.008867   
                    Shortest Path PCA                 1331.783471  0.008933   
Diversity Profile   Profile Shortest Path              189.983783  0.012069   
Embeddings          NetLSD                             573.590322  0.010561   
                    FEATHER                            511.824271  0.014100   
                    Spectral Zoo                       414.057220  0.016059   
                    Graph2Vec                          175.209092  0.018122   
Kernel              Kernel Shortest Path                      NaN  0.007883   
                    Kernel Shortest Path MDS          6421.048435  0.006249   
                    Kernel WL                                 NaN  0.007702   
                    Kernel WL MDS                     5360.941489  0.006252   
                    Kernel Core Framework MDS         1584.624929  0.012692   
                    Kernel Core Framework                     NaN  0.013915   
                    Kernel WL Optimal Assignment              NaN  0.001650   
                    Kernel WL Optimal Assignment MDS    48.285313  0.017901   
                    Kernel Pyramid Match                      NaN  0.007780   
                    Kernel Pyramid Match MDS             

## nrange30_100_stepsize5

In [27]:
# getting the right parameters
model_params = {
        'ER': {'ER_probs': [0.75], 
                'ER_nps': None,
        },
        'RPG':{
            'RPG_probs': [[0.9, 0.1]], 
            'RPG_props': [1/3, 1/3], 
        },
        'SBM': {
            'SBM_probs': [0.8, 0.05], 
            'SBM_blocks': [[1/4, 1/4, 1/4]]
            },
        'RG': {
            'RG_probs': [0.25]
        },
    }
    
models = list(model_params.keys())
all_configs = {}
for model in models:
    model_config = tuple(model_params[model][param] for param in model_params[model].keys())
    all_configs[model] = model_config
configuration = tuple(all_configs[model] for model in models)

# read dataframe
res_df = pd.read_csv(f'./simulations/results/sim_distributions/nrange30_100_stepsize5/res_df.csv', header=0, index_col=0)
#step compute cardinality: 5

# group methods
method_group = {}
for method in list(res_df.index):
   if 'toper' in method:
      method_group[method] = 'TopER'
   elif 'kernel' in method:
      method_group[method] = 'Kernel'
   elif 'embedding' in method or 'spectral_zoo' == method:
      method_group[method] = 'Embeddings'
   elif 'divcurves' in method:
      method_group[method] = 'Diversity Curves'
   elif 'div_function' in method or 'one_scale' in method:
      method_group[method] = 'Diversity Profile'
   elif 'persistent_homology' in method:
      method_group[method] = 'Persistent Homology'

      
res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
      'divcurves_shortest_path_distance': 'Shortest Path',
      'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
      'div_function_shortest_path_distance': 'Profile Shortest Path', 
      'graph_embedding_netLSD': 'NetLSD',
      'graph_embedding_graph2vec': 'Graph2Vec', 
      'graph_embedding_feathergraph': 'FEATHER',
      'kernel_shortest_path': 'Kernel Shortest Path', 
      'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
      'kernel_pyramid_match': 'Kernel Pyramid Match', 
      'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
      'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
      'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
      # 'kernel_weisfeiler_lehman': 'Kernel WL', out of memory error!
      # 'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
      'kernel_core_framework': 'Kernel Core Framework',
      'kernel_mds_core_framework': 'Kernel Core Framework MDS',
      'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
      'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
      'toper_degree_sub': 'TopER Degree', 
      'toper_popularity_sub': 'TopER Popularity', 
      'toper_forricci_sub': 'TopER Forricci',
      'toper_olricci_sub': 'TopER Olricci',
      'toper_closeness_sub': 'TopER Closeness', 
      'toper_all_sub': 'TopER All Filtrations',
      'spectral_zoo': 'Spectral Zoo', 
      'persistent_homology_degree': 'PH Degree',
      'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',
      'persistent_homology_forman_ricci_curvature': 'PH Fricci',
      'persistent_homology_laplacian': 'PH Laplacian', 
      'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
      'persistent_homology_alpha_complex': 'PH Alpha Complex',
}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)

# sort wrt to accuracy, in groups
df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)

scores = ['acc', 'silh', 'db', 'ch', 'ari_kmeans', 'ari_agglo']
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

res_copy = df.copy()

results_latex = {}

for score in scores:
   mean_col = f'{score}_mean'
   std_col = f'{score}_std'

   if mean_col in res_copy.columns and std_col in res_copy.columns:
      results_latex[score] = res_copy[mean_col].combine(
            res_copy[std_col],
            lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
      )

results_latex_df = pd.DataFrame.from_dict(results_latex)

# Bold the best mean for each score 
for score in scores:
   mean_col = f"{score}_mean"
   display_col =  score
   if mean_col in res_copy.columns:
      if not scores_asc[score]: 
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
      else:
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
      results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
            lambda x: f"\\textbf{{{x}}}"
      )
   else:
      print(f'{mean_col} not in dataframe?')

#final adjustments latex df 
results_latex_df.index.rename([None, 'Method'], inplace = True)
new_cols = {
   'acc': 'Accuracy', 
   'silh': 'Silhouette Score', 
   'db': 'Davies-Bouldin', 
   'ch': 'Calinski-Harabasz', 
   'ari_kmeans': 'ARI k-means', 
   'ari_agglo': 'ARI agglomerative'
}

results_latex_df = results_latex_df.rename(columns=new_cols)
results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

print(configuration)
print(results_latex_df.to_latex(multirow=False, escape=False))

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &           &                     Accuracy &             Silhouette Score &               Davies-Bouldin &                 Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                   &                              &                              \\
\midrule
Diversity Curves & Shortest Path &           0.998 $\pm $ 0.005 &           0.558 $\pm $ 0.012 &           0.591 $\pm $ 0.017 &             848.056 $\pm $ 60.991 &           0.567 $\pm $ 0.033 &           0.328 $\pm $ 0.132 \\
                    & Shortest Path PCA &           0.998 $\pm $ 0.005 &           0.570 $\pm $ 0.012 &           0.569 $\pm $ 0.017 &             870.549 $\pm $ 63.616 &    

In [28]:
df

silh_mean    db_mean  \
Group               Method                                                   
Diversity Curves    Shortest Path                      0.557585   0.590552   
                    Shortest Path PCA                  0.569829   0.568631   
Diversity Profile   Profile Shortest Path              0.092480   2.667957   
Embeddings          NetLSD                             0.310012   1.023193   
                    FEATHER                            0.142306   1.543725   
                    Spectral Zoo                       0.058362   1.380404   
                    Graph2Vec                         -0.067054   4.089604   
Kernel              Kernel Shortest Path               0.768010        NaN   
                    Kernel Core Framework              0.427237        NaN   
                    Kernel Shortest Path MDS           0.738012   0.394266   
                    Kernel Core Framework MDS          0.453802   0.815720   
                    Kernel WL Optimal Assignment       0.011398        NaN   
                    Kernel WL Optimal Assignment MDS   0.002027   5.259829   
                    Kernel Pyramid Match              -0.007579        NaN   
                    Kernel Pyramid Match MDS          -0.029449  24.649097   
                    Kernel Neighbourhood Hash         -0.024748        NaN   
                    Kernel Neighbourhood Hash MDS     -0.033969  62.341556   
TopER               TopER Degree                      -0.076262   2.445262   
                    TopER Closeness                   -0.080211   2.776251   
                    TopER All Filtrations             -0.045225   2.995246   
                    TopER Popularity                  -0.062747   6.469906   
                    TopER Olricci                     -0.087269   2.669990   
                    TopER Forricci                    -0.030448  17.649147   
Persistent Homology PH Vietoris Rips                   0.128168   2.280545   
                    PH Olricci                        -0.034432   1.392882   
                    PH Fricci                         -0.025309   1.551775   
                    PH Alpha Complex                   0.082735   5.259682   
                    PH Laplacian                      -0.135897   2.339386   
                    PH Degree                         -0.144335   2.411096   

                                                          ch_mean  silh_std  \
Group               Method                                                    
Diversity Curves    Shortest Path                      848.055923  0.011829   
                    Shortest Path PCA                  870.548774  0.012005   
Diversity Profile   Profile Shortest Path               87.817997  0.010629   
Embeddings          NetLSD                             199.323505  0.015752   
                    FEATHER                            238.349441  0.013917   
                    Spectral Zoo                       196.225465  0.014286   
                    Graph2Vec                           77.468174  0.014238   
Kernel              Kernel Shortest Path                      NaN  0.018793   
                    Kernel Core Framework                     NaN  0.012969   
                    Kernel Shortest Path MDS          3038.385342  0.013495   
                    Kernel Core Framework MDS          832.899920  0.014266   
                    Kernel WL Optimal Assignment              NaN  0.001368   
                    Kernel WL Optimal Assignment MDS    23.253114  0.008967   
                    Kernel Pyramid Match                      NaN  0.007412   
                    Kernel Pyramid Match MDS             2.301843  0.007798   
                    Kernel Neighbourhood Hash                 NaN  0.005290   
                    Kernel Neighbourhood Hash MDS        0.560784  0.006808   
TopER               TopER Degree                       113.610167  0.012032   
                    TopER Closeness                    

## nrange10_100_stepsize5_ngraphs1

In [29]:
# getting the right parameters
model_params = {
        'ER': {'ER_probs': [0.75], 
                'ER_nps': None,
        },
        'RPG':{
            'RPG_probs': [[0.9, 0.1]], 
            'RPG_props': [1/3, 1/3], 
        },
        'SBM': {
            'SBM_probs': [0.8, 0.05], 
            'SBM_blocks': [[1/4, 1/4, 1/4]]
            },
        'RG': {
            'RG_probs': [0.25]
        },
    }
    
models = list(model_params.keys())
all_configs = {}
for model in models:
    model_config = tuple(model_params[model][param] for param in model_params[model].keys())
    all_configs[model] = model_config
configuration = tuple(all_configs[model] for model in models)

# read dataframe
res_df = pd.read_csv(f'./simulations/results/sim_distributions/nrange10_100_stepsize5_ngraphs1/res_df.csv', header=0, index_col=0)

# group methods
method_group = {}
for method in list(res_df.index):
   if 'toper' in method:
      method_group[method] = 'TopER'
   elif 'kernel' in method:
      method_group[method] = 'Kernel'
   elif 'embedding' in method or 'spectral_zoo' == method:
      method_group[method] = 'Embeddings'
   elif 'divcurves' in method:
      method_group[method] = 'Diversity Curves'
   elif 'div_function' in method or 'one_scale' in method:
      method_group[method] = 'Diversity Profile'
   elif 'persistent_homology' in method:
      method_group[method] = 'Persistent Homology'

      
res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
      'divcurves_shortest_path_distance': 'Shortest Path',
      'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
      'div_function_shortest_path_distance': 'Profile Shortest Path', 
      'graph_embedding_netLSD': 'NetLSD',
      'graph_embedding_graph2vec': 'Graph2Vec', 
      'graph_embedding_feathergraph': 'FEATHER',
      'kernel_shortest_path': 'Kernel Shortest Path', 
      'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
      'kernel_pyramid_match': 'Kernel Pyramid Match', 
      'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
      'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
      'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
      # 'kernel_weisfeiler_lehman': 'Kernel WL', out of memory error!
      # 'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
      'kernel_core_framework': 'Kernel Core Framework',
      'kernel_mds_core_framework': 'Kernel Core Framework MDS',
      'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
      'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
      'toper_degree_sub': 'TopER Degree', 
      'toper_popularity_sub': 'TopER Popularity', 
      'toper_forricci_sub': 'TopER Forricci',
      'toper_olricci_sub': 'TopER Olricci',
      'toper_closeness_sub': 'TopER Closeness', 
      'toper_all_sub': 'TopER All Filtrations',
      'spectral_zoo': 'Spectral Zoo', 
      'persistent_homology_degree': 'PH Degree',
      'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',
      'persistent_homology_forman_ricci_curvature': 'PH Fricci',
      'persistent_homology_laplacian': 'PH Laplacian', 
      'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
      'persistent_homology_alpha_complex': 'PH Alpha Complex',
}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)

# sort wrt to accuracy, in groups
df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)

scores = ['acc', 'silh', 'db', 'ch', 'ari_kmeans', 'ari_agglo']
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

res_copy = df.copy()

results_latex = {}

for score in scores:
   mean_col = f'{score}_mean'
   std_col = f'{score}_std'

   if mean_col in res_copy.columns and std_col in res_copy.columns:
      results_latex[score] = res_copy[mean_col].combine(
            res_copy[std_col],
            lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
      )

results_latex_df = pd.DataFrame.from_dict(results_latex)

# Bold the best mean for each score 
for score in scores:
   mean_col = f"{score}_mean"
   display_col =  score
   if mean_col in res_copy.columns:
      if not scores_asc[score]: 
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
      else:
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
      results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
            lambda x: f"\\textbf{{{x}}}"
      )
   else:
      print(f'{mean_col} not in dataframe?')

#final adjustments latex df 
results_latex_df.index.rename([None, 'Method'], inplace = True)
new_cols = {
   'acc': 'Accuracy', 
   'silh': 'Silhouette Score', 
   'db': 'Davies-Bouldin', 
   'ch': 'Calinski-Harabasz', 
   'ari_kmeans': 'ARI k-means', 
   'ari_agglo': 'ARI agglomerative'
}

results_latex_df = results_latex_df.rename(columns=new_cols)
results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

print(configuration)
print(results_latex_df.to_latex(multirow=False, escape=False))

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &           &                     Accuracy &             Silhouette Score &               Davies-Bouldin &               Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                 &                              &                              \\
\midrule
Diversity Curves & Shortest Path &  \textbf{0.989 $\pm $ 0.018} &           0.400 $\pm $ 0.027 &           0.841 $\pm $ 0.057 &           174.464 $\pm $ 20.701 &           0.452 $\pm $ 0.116 &           0.239 $\pm $ 0.019 \\
                    & Shortest Path PCA &           0.981 $\pm $ 0.022 &           0.412 $\pm $ 0.028 &  \textbf{0.815 $\pm $ 0.058} &           177.461 $\pm $ 21.321 &           0

In [30]:
df

silh_mean    db_mean  \
Group               Method                                                   
Diversity Curves    Shortest Path                      0.399547   0.841142   
                    Shortest Path PCA                  0.412274   0.814556   
Diversity Profile   Profile Shortest Path             -0.005707   4.696204   
Embeddings          NetLSD                            -0.080270   2.254459   
                    FEATHER                            0.018777   2.930202   
                    Spectral Zoo                      -0.090816   3.550703   
                    Graph2Vec                         -0.115299   8.947503   
Kernel              Kernel WL Optimal Assignment       0.001226        NaN   
                    Kernel Core Framework              0.244863        NaN   
                    Kernel Shortest Path MDS           0.507928   0.935320   
                    Kernel Shortest Path               0.510584        NaN   
                    Kernel Core Framework MDS          0.231303   1.666085   
                    Kernel WL Optimal Assignment MDS  -0.018208   9.565093   
                    Kernel Pyramid Match              -0.041312        NaN   
                    Kernel Pyramid Match MDS          -0.053850  22.039970   
                    Kernel Neighbourhood Hash         -0.041833        NaN   
                    Kernel Neighbourhood Hash MDS     -0.056516  33.344124   
TopER               TopER All Filtrations             -0.105813   4.436944   
                    TopER Degree                      -0.137318   4.399021   
                    TopER Popularity                  -0.126508   9.172269   
                    TopER Closeness                   -0.131140   4.888090   
                    TopER Forricci                    -0.085365  29.657580   
                    TopER Olricci                     -0.119050   8.963265   
Persistent Homology PH Vietoris Rips                   0.021350   2.847709   
                    PH Olricci                        -0.143917   1.788545   
                    PH Alpha Complex                   0.007277  10.148593   
                    PH Fricci                         -0.122211   1.769898   
                    PH Laplacian                      -0.207238   3.003925   
                    PH Degree                         -0.208338   2.730104   

                                                         ch_mean  silh_std  \
Group               Method                                                   
Diversity Curves    Shortest Path                     174.464398  0.027053   
                    Shortest Path PCA                 177.460755  0.027736   
Diversity Profile   Profile Shortest Path              13.729837  0.016089   
Embeddings          NetLSD                             13.848358  0.048585   
                    FEATHER                            32.490323  0.017559   
                    Spectral Zoo                       17.556630  0.028705   
                    Graph2Vec                          10.059034  0.014090   
Kernel              Kernel WL Optimal Assignment             NaN  0.001824   
                    Kernel Core Framework                    NaN  0.019798   
                    Kernel Shortest Path MDS          180.706528  0.033547   
                    Kernel Shortest Path                     NaN  0.039253   
                    Kernel Core Framework MDS          89.373064  0.023990   
                    Kernel WL Optimal Assignment MDS    6.546438  0.010429   
                    Kernel Pyramid Match                     NaN  0.013114   
                    Kernel Pyramid Match MDS            0.999971  0.013222   
                    Kernel Neighbourhood Hash                NaN  0.009254   
                    Kernel Neighbourhood Hash MDS       0.571578  0.011305   
TopER               TopER All Filtrations              23.680429  0.015913   
                    TopER Degree                       24.777677  0.020004 

## nrange10_30_and1030_and50_80_stepcard_3

In [31]:
# getting the right parameters
model_params = {
        'ER': {'ER_probs': [0.75], 
                'ER_nps': None,
        },
        'RPG':{
            'RPG_probs': [[0.9, 0.1]], 
            'RPG_props': [1/3, 1/3], 
        },
        'SBM': {
            'SBM_probs': [0.8, 0.05], 
            'SBM_blocks': [[1/4, 1/4, 1/4]]
            },
        'RG': {
            'RG_probs': [0.25]
        },
    }
    
models = list(model_params.keys())
all_configs = {}
for model in models:
    model_config = tuple(model_params[model][param] for param in model_params[model].keys())
    all_configs[model] = model_config
configuration = tuple(all_configs[model] for model in models)

# read dataframe
res_df = pd.read_csv(f'simulations/results/sim_distributions/nrange10_30_and1030_and50_80_stepcard_3/res_df.csv', header=0, index_col=0)

# group methods
method_group = {}
for method in list(res_df.index):
   if 'toper' in method:
      method_group[method] = 'TopER'
   elif 'kernel' in method:
      method_group[method] = 'Kernel'
   elif 'embedding' in method or 'spectral_zoo' == method:
      method_group[method] = 'Embeddings'
   elif 'divcurves' in method:
      method_group[method] = 'Diversity Curves'
   elif 'div_function' in method or 'one_scale' in method:
      method_group[method] = 'Diversity Profile'
   elif 'persistent_homology' in method:
      method_group[method] = 'Persistent Homology'

      
res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
      'divcurves_shortest_path_distance': 'Shortest Path',
      'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
      'div_function_shortest_path_distance': 'Profile Shortest Path', 
      'graph_embedding_netLSD': 'NetLSD',
      'graph_embedding_graph2vec': 'Graph2Vec', 
      'graph_embedding_feathergraph': 'FEATHER',
      'kernel_shortest_path': 'Kernel Shortest Path', 
      'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
      'kernel_pyramid_match': 'Kernel Pyramid Match', 
      'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
      'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
      'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
      # 'kernel_weisfeiler_lehman': 'Kernel WL', out of memory error!
      # 'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
      'kernel_core_framework': 'Kernel Core Framework',
      'kernel_mds_core_framework': 'Kernel Core Framework MDS',
      'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
      'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
      'toper_degree_sub': 'TopER Degree', 
      'toper_popularity_sub': 'TopER Popularity', 
      'toper_forricci_sub': 'TopER Forricci',
      'toper_olricci_sub': 'TopER Olricci',
      'toper_closeness_sub': 'TopER Closeness', 
      'toper_all_sub': 'TopER All Filtrations',
      'spectral_zoo': 'Spectral Zoo', 
      'persistent_homology_degree': 'PH Degree',
      'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',
      'persistent_homology_forman_ricci_curvature': 'PH Fricci',
      'persistent_homology_laplacian': 'PH Laplacian', 
      'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
      'persistent_homology_alpha_complex': 'PH Alpha Complex',
}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)

# sort wrt to accuracy, in groups
df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)

scores = ['acc', 'silh', 'db', 'ch', 'ari_kmeans', 'ari_agglo']
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

res_copy = df.copy()

results_latex = {}

for score in scores:
   mean_col = f'{score}_mean'
   std_col = f'{score}_std'

   if mean_col in res_copy.columns and std_col in res_copy.columns:
      results_latex[score] = res_copy[mean_col].combine(
            res_copy[std_col],
            lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
      )

results_latex_df = pd.DataFrame.from_dict(results_latex)

# Bold the best mean for each score 
for score in scores:
   mean_col = f"{score}_mean"
   display_col =  score
   if mean_col in res_copy.columns:
      if not scores_asc[score]: 
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
      else:
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
      results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
            lambda x: f"\\textbf{{{x}}}"
      )
   else:
      print(f'{mean_col} not in dataframe?')

#final adjustments latex df 
results_latex_df.index.rename([None, 'Method'], inplace = True)
new_cols = {
   'acc': 'Accuracy', 
   'silh': 'Silhouette Score', 
   'db': 'Davies-Bouldin', 
   'ch': 'Calinski-Harabasz', 
   'ari_kmeans': 'ARI k-means', 
   'ari_agglo': 'ARI agglomerative'
}

results_latex_df = results_latex_df.rename(columns=new_cols)
results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

print(configuration)
print(results_latex_df.to_latex(multirow=False, escape=False))

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &                  &                     Accuracy &             Silhouette Score &               Davies-Bouldin &              Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                &                              &                              \\
\midrule
Diversity Curves & Shortest Path &           0.830 $\pm $ 0.051 &           0.017 $\pm $ 0.044 &           1.799 $\pm $ 0.182 &            29.437 $\pm $ 5.177 &           0.214 $\pm $ 0.037 &           0.122 $\pm $ 0.092 \\
                    & Shortest Path PCA &           0.775 $\pm $ 0.081 &           0.015 $\pm $ 0.045 &           1.790 $\pm $ 0.182 &            29.566 $\pm $ 5.210 &         

In [32]:
df

silh_mean    db_mean  \
Group               Method                                                   
Diversity Curves    Shortest Path                      0.017056   1.798639   
                    Shortest Path PCA                  0.014550   1.789618   
Diversity Profile   Profile Shortest Path             -0.005691   5.596967   
Embeddings          NetLSD                            -0.129037   2.513782   
                    FEATHER                            0.047696   3.575875   
                    Spectral Zoo                      -0.118685   5.586805   
                    Graph2Vec                         -0.023541   6.883303   
Kernel              Kernel WL Optimal Assignment       0.009470        NaN   
                    Kernel Shortest Path               0.378774        NaN   
                    Kernel Core Framework MDS          0.217752   2.139959   
                    Kernel Shortest Path MDS           0.396276   1.396062   
                    Kernel Core Framework              0.216217        NaN   
                    Kernel WL Optimal Assignment MDS  -0.007663   8.562435   
                    Kernel Pyramid Match              -0.064061        NaN   
                    Kernel Pyramid Match MDS          -0.080039  18.470194   
                    Kernel Neighbourhood Hash         -0.060020        NaN   
                    Kernel Neighbourhood Hash MDS     -0.080602  26.098712   
TopER               TopER All Filtrations             -0.082698   5.073878   
                    TopER Degree                      -0.104632   5.918247   
                    TopER Popularity                  -0.093159  12.288111   
                    TopER Closeness                   -0.101656   6.985245   
                    TopER Forricci                    -0.087645  15.104988   
                    TopER Olricci                     -0.100540  13.176657   
Persistent Homology PH Vietoris Rips                   0.055770   2.384412   
                    PH Laplacian                      -0.158402   2.883307   
                    PH Fricci                         -0.067281   1.726257   
                    PH Olricci                        -0.083930   1.728999   
                    PH Degree                         -0.169863   2.547801   
                    PH Alpha Complex                  -0.002866   9.649223   

                                                        ch_mean  silh_std  \
Group               Method                                                  
Diversity Curves    Shortest Path                     29.436826  0.043981   
                    Shortest Path PCA                 29.565819  0.044699   
Diversity Profile   Profile Shortest Path              6.145569  0.026332   
Embeddings          NetLSD                             9.148898  0.048752   
                    FEATHER                           14.933975  0.028378   
                    Spectral Zoo                       9.896217  0.038757   
                    Graph2Vec                          1.486386  0.003803   
Kernel              Kernel WL Optimal Assignment            NaN  0.003239   
                    Kernel Shortest Path                    NaN  0.049942   
                    Kernel Core Framework MDS         39.260082  0.030857   
                    Kernel Shortest Path MDS          57.180903  0.041161   
                    Kernel Core Framework                   NaN  0.028911   
                    Kernel WL Optimal Assignment MDS   3.830912  0.015435   
                    Kernel Pyramid Match                    NaN  0.018593   
                    Kernel Pyramid Match MDS           0.773068  0.019651   
                    Kernel Neighbourhood Hash               NaN  0.013942   
                    Kernel Neighbourhood Hash MDS      0.531593  0.016432   
TopER               TopER All Filtrations             14.142890  0.024594   
                    TopER Degree                      14.610599  0.029462   
                  

## nrange10_30_and100_130_stepcard_3

In [33]:
# getting the right parameters
model_params = {
        'ER': {'ER_probs': [0.75], 
                'ER_nps': None,
        },
        'RPG':{
            'RPG_probs': [[0.9, 0.1]], 
            'RPG_props': [1/3, 1/3], 
        },
        'SBM': {
            'SBM_probs': [0.8, 0.05], 
            'SBM_blocks': [[1/4, 1/4, 1/4]]
            },
        'RG': {
            'RG_probs': [0.25]
        },
    }
    
models = list(model_params.keys())
all_configs = {}
for model in models:
    model_config = tuple(model_params[model][param] for param in model_params[model].keys())
    all_configs[model] = model_config
configuration = tuple(all_configs[model] for model in models)

# read dataframe
res_df = pd.read_csv(f'simulations/results/sim_distributions/nrange10_30_and100_130_stepcard_3/res_df.csv', header=0, index_col=0)

# group methods
method_group = {}
for method in list(res_df.index):
   if 'toper' in method:
      method_group[method] = 'TopER'
   elif 'kernel' in method:
      method_group[method] = 'Kernel'
   elif 'embedding' in method or 'spectral_zoo' == method:
      method_group[method] = 'Embeddings'
   elif 'divcurves' in method:
      method_group[method] = 'Diversity Curves'
   elif 'div_function' in method or 'one_scale' in method:
      method_group[method] = 'Diversity Profile'
   elif 'persistent_homology' in method:
      method_group[method] = 'Persistent Homology'

      
res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
      'divcurves_shortest_path_distance': 'Shortest Path',
      'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
      'div_function_shortest_path_distance': 'Profile Shortest Path', 
      'graph_embedding_netLSD': 'NetLSD',
      'graph_embedding_graph2vec': 'Graph2Vec', 
      'graph_embedding_feathergraph': 'FEATHER',
      'kernel_shortest_path': 'Kernel Shortest Path', 
      'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
      'kernel_pyramid_match': 'Kernel Pyramid Match', 
      'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
      'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
      'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
      # 'kernel_weisfeiler_lehman': 'Kernel WL', out of memory error!
      # 'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
      'kernel_core_framework': 'Kernel Core Framework',
      'kernel_mds_core_framework': 'Kernel Core Framework MDS',
      'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
      'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
      'toper_degree_sub': 'TopER Degree', 
      'toper_popularity_sub': 'TopER Popularity', 
      'toper_forricci_sub': 'TopER Forricci',
      'toper_olricci_sub': 'TopER Olricci',
      'toper_closeness_sub': 'TopER Closeness', 
      'toper_all_sub': 'TopER All Filtrations',
      'spectral_zoo': 'Spectral Zoo', 
      'persistent_homology_degree': 'PH Degree',
      'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',
      'persistent_homology_forman_ricci_curvature': 'PH Fricci',
      'persistent_homology_laplacian': 'PH Laplacian', 
      'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
      'persistent_homology_alpha_complex': 'PH Alpha Complex',
}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)

# sort wrt to accuracy, in groups
df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)

scores = ['acc', 'silh', 'db', 'ch', 'ari_kmeans', 'ari_agglo']
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

res_copy = df.copy()

results_latex = {}

for score in scores:
   mean_col = f'{score}_mean'
   std_col = f'{score}_std'

   if mean_col in res_copy.columns and std_col in res_copy.columns:
      results_latex[score] = res_copy[mean_col].combine(
            res_copy[std_col],
            lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
      )

results_latex_df = pd.DataFrame.from_dict(results_latex)

# Bold the best mean for each score 
for score in scores:
   mean_col = f"{score}_mean"
   display_col =  score
   if mean_col in res_copy.columns:
      if not scores_asc[score]: 
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
      else:
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
      results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
            lambda x: f"\\textbf{{{x}}}"
      )
   else:
      print(f'{mean_col} not in dataframe?')

#final adjustments latex df 
results_latex_df.index.rename([None, 'Method'], inplace = True)
new_cols = {
   'acc': 'Accuracy', 
   'silh': 'Silhouette Score', 
   'db': 'Davies-Bouldin', 
   'ch': 'Calinski-Harabasz', 
   'ari_kmeans': 'ARI k-means', 
   'ari_agglo': 'ARI agglomerative'
}

results_latex_df = results_latex_df.rename(columns=new_cols)
results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

print(configuration)
print(results_latex_df.to_latex(multirow=False, escape=False))

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &           &                     Accuracy &             Silhouette Score &               Davies-Bouldin &              Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                &                              &                              \\
\midrule
Diversity Curves & Shortest Path &           0.880 $\pm $ 0.056 &          -0.124 $\pm $ 0.053 &           2.805 $\pm $ 0.576 &            14.848 $\pm $ 3.365 &           0.197 $\pm $ 0.079 &           0.052 $\pm $ 0.026 \\
                    & Shortest Path PCA &           0.865 $\pm $ 0.050 &          -0.124 $\pm $ 0.053 &           2.802 $\pm $ 0.583 &            14.882 $\pm $ 3.377 &           0.200

In [34]:
df

silh_mean    db_mean  \
Group               Method                                                   
Diversity Curves    Shortest Path                     -0.124161   2.805297   
                    Shortest Path PCA                 -0.124248   2.802133   
Diversity Profile   Profile Shortest Path              0.021031   6.736413   
Embeddings          NetLSD                            -0.189568   2.999869   
                    FEATHER                            0.103627   4.253546   
                    Spectral Zoo                      -0.172388  11.059262   
                    Graph2Vec                         -0.068156   5.756658   
Kernel              Kernel Core Framework MDS          0.209711   2.956793   
                    Kernel WL Optimal Assignment       0.021881        NaN   
                    Kernel Core Framework              0.215387        NaN   
                    Kernel Shortest Path               0.331518        NaN   
                    Kernel Shortest Path MDS           0.382636   1.869944   
                    Kernel WL Optimal Assignment MDS   0.000642   6.961924   
                    Kernel Pyramid Match              -0.090403        NaN   
                    Kernel Pyramid Match MDS          -0.102842  30.354232   
                    Kernel Neighbourhood Hash MDS     -0.091341  30.901951   
                    Kernel Neighbourhood Hash         -0.074596        NaN   
TopER               TopER All Filtrations             -0.018275   4.535771   
                    TopER Popularity                  -0.016734   7.197992   
                    TopER Degree                      -0.060075   7.536136   
                    TopER Closeness                   -0.061664   7.789758   
                    TopER Forricci                    -0.078091  45.193356   
                    TopER Olricci                     -0.117292  20.245287   
Persistent Homology PH Vietoris Rips                   0.064828   3.913042   
                    PH Laplacian                      -0.057590   1.880431   
                    PH Olricci                         0.029536   1.523407   
                    PH Fricci                          0.041912   1.470420   
                    PH Alpha Complex                   0.030988   4.101760   
                    PH Degree                         -0.033060   1.597921   

                                                        ch_mean  silh_std  \
Group               Method                                                  
Diversity Curves    Shortest Path                     14.847826  0.052629   
                    Shortest Path PCA                 14.882409  0.053035   
Diversity Profile   Profile Shortest Path              5.409481  0.035053   
Embeddings          NetLSD                             7.761336  0.044329   
                    FEATHER                            7.978733  0.030790   
                    Spectral Zoo                       6.172980  0.035286   
                    Graph2Vec                          4.288678  0.016182   
Kernel              Kernel Core Framework MDS         24.195478  0.030757   
                    Kernel WL Optimal Assignment            NaN  0.003996   
                    Kernel Core Framework                   NaN  0.029016   
                    Kernel Shortest Path                    NaN  0.049350   
                    Kernel Shortest Path MDS          50.872391  0.042648   
                    Kernel WL Optimal Assignment MDS   3.591337  0.018063   
                    Kernel Pyramid Match                    NaN  0.025190   
                    Kernel Pyramid Match MDS           0.527537  0.025820   
                    Kernel Neighbourhood Hash MDS      0.517465  0.020652   
                    Kernel Neighbourhood Hash               NaN  0.019146   
TopER               TopER All Filtrations             12.565420  0.033336   
                    TopER Popularity                  13.086210  0.038752   
                  

## test checkpoint

In [5]:
# getting the right parameters
model_params = {
        'ER': {'ER_probs': [0.75], 
                'ER_nps': None,
        },
        'RPG':{
            'RPG_probs': [[0.9, 0.1]], 
            'RPG_props': [1/3, 1/3], 
        },
        'SBM': {
            'SBM_probs': [0.8, 0.05], 
            'SBM_blocks': [[1/4, 1/4, 1/4]]
            },
        'RG': {
            'RG_probs': [0.25]
        },
    }
    
models = list(model_params.keys())
all_configs = {}
for model in models:
    model_config = tuple(model_params[model][param] for param in model_params[model].keys())
    all_configs[model] = model_config
configuration = tuple(all_configs[model] for model in models)

# read dataframe
res_df = pd.read_csv(f'simulations/results/sim_distributions/nrange10_30_checjpoint_test/res_df.csv', header=0, index_col=0)

# group methods
method_group = {}
for method in list(res_df.index):
   if 'toper' in method:
      method_group[method] = 'TopER'
   elif 'kernel' in method:
      method_group[method] = 'Kernel'
   elif 'embedding' in method or 'spectral_zoo' == method:
      method_group[method] = 'Embeddings'
   elif 'divcurves' in method:
      method_group[method] = 'Diversity Curves'
   elif 'div_function' in method or 'one_scale' in method:
      method_group[method] = 'Diversity Profile'
   elif 'persistent_homology' in method:
      method_group[method] = 'Persistent Homology'

      
res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
      'divcurves_shortest_path_distance': 'Shortest Path',
      'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
      'div_function_shortest_path_distance': 'Profile Shortest Path', 
      'graph_embedding_netLSD': 'NetLSD',
      'graph_embedding_graph2vec': 'Graph2Vec', 
      'graph_embedding_feathergraph': 'FEATHER',
      'kernel_shortest_path': 'Kernel Shortest Path', 
      'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
      'kernel_pyramid_match': 'Kernel Pyramid Match', 
      'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
      'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
      'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
      # 'kernel_weisfeiler_lehman': 'Kernel WL', out of memory error!
      # 'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
    #   'kernel_core_framework': 'Kernel Core Framework',
    #   'kernel_mds_core_framework': 'Kernel Core Framework MDS',
    #   'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
    #   'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
      'toper_degree_sub': 'TopER Degree', 
      'toper_popularity_sub': 'TopER Popularity', 
      'toper_forricci_sub': 'TopER Forricci',
      'toper_olricci_sub': 'TopER Olricci',
      'toper_closeness_sub': 'TopER Closeness', 
      'toper_all_sub': 'TopER All Filtrations',
      'spectral_zoo': 'Spectral Zoo', 
      'persistent_homology_degree': 'PH Degree',
      'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',
      'persistent_homology_forman_ricci_curvature': 'PH Fricci',
      'persistent_homology_laplacian': 'PH Laplacian', 
      'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
      'persistent_homology_alpha_complex': 'PH Alpha Complex',
}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)

# sort wrt to accuracy, in groups
df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)

scores = ['acc', 'silh', 'db', 'ch', 'ari_kmeans', 'ari_agglo']
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

res_copy = df.copy()

results_latex = {}

for score in scores:
   mean_col = f'{score}_mean'
   std_col = f'{score}_std'

   if mean_col in res_copy.columns and std_col in res_copy.columns:
      results_latex[score] = res_copy[mean_col].combine(
            res_copy[std_col],
            lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
      )

results_latex_df = pd.DataFrame.from_dict(results_latex)

# Bold the best mean for each score 
for score in scores:
   mean_col = f"{score}_mean"
   display_col =  score
   if mean_col in res_copy.columns:
      if not scores_asc[score]: 
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
      else:
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
      results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
            lambda x: f"\\textbf{{{x}}}"
      )
   else:
      print(f'{mean_col} not in dataframe?')

#final adjustments latex df 
results_latex_df.index.rename([None, 'Method'], inplace = True)
new_cols = {
   'acc': 'Accuracy', 
   'silh': 'Silhouette Score', 
   'db': 'Davies-Bouldin', 
   'ch': 'Calinski-Harabasz', 
   'ari_kmeans': 'ARI k-means', 
   'ari_agglo': 'ARI agglomerative'
}

results_latex_df = results_latex_df.rename(columns=new_cols)
results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

print(configuration)
print(results_latex_df.to_latex(multirow=False, escape=False))

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &            &                     Accuracy &             Silhouette Score &               Davies-Bouldin &               Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                 &                              &                              \\
\midrule
Diversity Curves & Shortest Path &  \textbf{0.900 $\pm $ 0.046} &           0.420 $\pm $ 0.034 &           0.759 $\pm $ 0.061 &           190.603 $\pm $ 33.487 &  \textbf{0.504 $\pm $ 0.079} &           0.312 $\pm $ 0.117 \\
                    & Shortest Path PCA &           0.892 $\pm $ 0.053 &  \textbf{0.429 $\pm $ 0.035} &  \textbf{0.743 $\pm $ 0.061} &  \textbf{193.586 $\pm $ 34.412} &           

In [6]:
df

silh_mean    db_mean  \
Group               Method                                                
Diversity Curves    Shortest Path                   0.419790   0.758842   
                    Shortest Path PCA               0.428732   0.742730   
Diversity Profile   Profile Shortest Path           0.032804   3.948620   
Embeddings          NetLSD                          0.055250   1.332243   
                    FEATHER                         0.107860   1.914224   
                    Spectral Zoo                    0.050488   2.874773   
                    Graph2Vec                      -0.004701   7.375717   
Kernel              Kernel Shortest Path            0.284192        NaN   
                    Kernel Shortest Path MDS        0.273800   1.594493   
                    Kernel Pyramid Match            0.001550        NaN   
                    Kernel Pyramid Match MDS       -0.006689   6.301723   
                    Kernel Neighbourhood Hash      -0.031523        NaN   
                    Kernel Neighbourhood Hash MDS  -0.066606  16.320215   
TopER               TopER Degree                   -0.064163   5.288275   
                    TopER Closeness                -0.092908   8.445117   
                    TopER All Filtrations          -0.055451   3.254579   
                    TopER Popularity               -0.022413   1.343662   
                    TopER Forricci                 -0.066691   2.950042   
                    TopER Olricci                  -0.098534  10.200269   
Persistent Homology PH Vietoris Rips               -0.010201   2.226599   
                    PH Laplacian                   -0.131984   2.668183   
                    PH Alpha Complex                0.014232   1.992017   
                    PH Fricci                      -0.099454   2.498226   
                    PH Degree                      -0.159543   3.015931   
                    PH Olricci                     -0.124737   2.944712   

                                                      ch_mean  silh_std  \
Group               Method                                                
Diversity Curves    Shortest Path                  190.602869  0.033826   
                    Shortest Path PCA              193.585742  0.034513   
Diversity Profile   Profile Shortest Path           12.516997  0.019242   
Embeddings          NetLSD                          38.798588  0.037856   
                    FEATHER                         74.718701  0.023478   
                    Spectral Zoo                    33.793833  0.030683   
                    Graph2Vec                        1.409031  0.001397   
Kernel              Kernel Shortest Path                  NaN  0.043685   
                    Kernel Shortest Path MDS        57.119353  0.038198   
                    Kernel Pyramid Match                  NaN  0.008959   
                    Kernel Pyramid Match MDS         7.472089  0.011575   
                    Kernel Neighbourhood Hash             NaN  0.005716   
                    Kernel Neighbourhood Hash MDS    0.994753  0.007074   
TopER               TopER Degree                    31.190819  0.021400   
                    TopER Closeness                 25.481925  0.018945   
                    TopER All Filtrations           27.543609  0.019831   
                    TopER Popularity                29.667016  0.037472   
                    TopER Forricci                  31.449990  0.022240   
                    TopER Olricci                   13.488252  0.015377   
Persistent Homology PH Vietoris Rips                16.155757  0.030998   
                    PH Laplacian                    11.701227  0.018462   
                    PH Alpha Complex                43.715269  0.020705   
                    PH Fricci                       30.873838  0.020466   
                    PH Degree                        8.580410  0.019224   
                    PH Olricci                      34.161159  0.023068   


## only one cardinality, but same number of the classes

In [13]:
# getting the right parameters
model_params = {
        'ER': {'ER_probs': [0.75], 
                'ER_nps': None,
        },
        'RPG':{
            'RPG_probs': [[0.9, 0.1]], 
            'RPG_props': [1/3, 1/3], 
        },
        'SBM': {
            'SBM_probs': [0.8, 0.05], 
            'SBM_blocks': [[1/4, 1/4, 1/4]]
            },
        'RG': {
            'RG_probs': [0.25]
        },
    }
    
models = list(model_params.keys())
all_configs = {}
for model in models:
    model_config = tuple(model_params[model][param] for param in model_params[model].keys())
    all_configs[model] = model_config
configuration = tuple(all_configs[model] for model in models)

# read dataframe
res_df = pd.read_csv(f'./simulations/results/sim_distributions/one_cardinality/res_df.csv', header=0, index_col=0)


# group methods
method_group = {}
for method in list(res_df.index):
   if 'toper' in method:
      method_group[method] = 'TopER'
   elif 'kernel' in method:
      method_group[method] = 'Kernel'
   elif 'embedding' in method or 'spectral_zoo' == method:
      method_group[method] = 'Embeddings'
   elif 'divcurves' in method:
      method_group[method] = 'Diversity Curves'
   elif 'div_function' in method or 'one_scale' in method:
      method_group[method] = 'Diversity Profile'
   elif 'persistent_homology' in method:
      method_group[method] = 'Persistent Homology'

      
res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
      'divcurves_shortest_path_distance': 'Shortest Path',
      # 'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
      # 'div_function_shortest_path_distance': 'Profile Shortest Path', 
      'graph_embedding_netLSD': 'NetLSD',
      # 'graph_embedding_graph2vec': 'Graph2Vec', 
      'graph_embedding_feathergraph': 'FEATHER',
      'kernel_shortest_path': 'Kernel Shortest Path', 
      # 'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
      # 'kernel_pyramid_match': 'Kernel Pyramid Match', 
      # 'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
      # 'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
      # 'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
      # 'kernel_weisfeiler_lehman': 'Kernel WL',
      # 'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
      # 'kernel_core_framework': 'Kernel Core Framework',
      # 'kernel_mds_core_framework': 'Kernel Core Framework MDS',
      'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
      # 'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
      # 'toper_degree_sub': 'TopER Degree', 
      # 'toper_popularity_sub': 'TopER Popularity', 
      # 'toper_forricci_sub': 'TopER Forricci',
      # 'toper_olricci_sub': 'TopER Olricci',
      # 'toper_closeness_sub': 'TopER Closeness', 
      'toper_all_sub': 'TopER All Filtrations',
      'spectral_zoo': 'Spectral Zoo', 
      # 'persistent_homology_degree': 'PH Degree',
      # 'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',
      # 'persistent_homology_forman_ricci_curvature': 'PH Fricci',
      # 'persistent_homology_laplacian': 'PH Laplacian', 
      'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
      # 'persistent_homology_alpha_complex': 'PH Alpha Complex',
}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)

# sort wrt to accuracy, in groups
df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)

scores = [
   'acc', 
   'silh', 
   # 'db', 
   # 'ch', 
   # 'ari_kmeans', 
   # 'ari_agglo'
   ]
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

res_copy = df.copy()

results_latex = {}

for score in scores:
   mean_col = f'{score}_mean'
   std_col = f'{score}_std'

   if mean_col in res_copy.columns and std_col in res_copy.columns:
      results_latex[score] = res_copy[mean_col].combine(
            res_copy[std_col],
            lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
      )

results_latex_df = pd.DataFrame.from_dict(results_latex)

# Bold the best mean for each score 
for score in scores:
   mean_col = f"{score}_mean"
   display_col =  score
   if mean_col in res_copy.columns:
      if not scores_asc[score]: 
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
      else:
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
      results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
            lambda x: f"\\textbf{{{x}}}"
      )
   else:
      print(f'{mean_col} not in dataframe?')

#final adjustments latex df 
results_latex_df.index.rename([None, 'Method'], inplace = True)
new_cols = {
   'acc': 'Accuracy', 
   'silh': 'Silhouette Score', 
   'db': 'Davies-Bouldin', 
   'ch': 'Calinski-Harabasz', 
   'ari_kmeans': 'ARI k-means', 
   'ari_agglo': 'ARI agglomerative'
}

results_latex_df = results_latex_df.rename(columns=new_cols)
results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

print(configuration)
print(results_latex_df.to_latex(multirow=False, escape=False))

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llll}
\toprule
                    &                  &                     Accuracy &             Silhouette Score \\
{} & Method &                              &                              \\
\midrule
Diversity Curves & Shortest Path &           0.992 $\pm $ 0.017 &           0.753 $\pm $ 0.019 \\
Embeddings & FEATHER &  \textbf{0.996 $\pm $ 0.012} &           0.688 $\pm $ 0.019 \\
                    & NetLSD &           0.979 $\pm $ 0.021 &           0.648 $\pm $ 0.028 \\
                    & Spectral Zoo &           0.979 $\pm $ 0.021 &           0.675 $\pm $ 0.024 \\
Kernel & Kernel WL Optimal Assignment &  \textbf{0.996 $\pm $ 0.012} &           0.089 $\pm $ 0.003 \\
                    & Kernel Shortest Path &           0.988 $\pm $ 0.019 &  \textbf{0.772 $\pm $ 0.025} \\
TopER & TopER All Filtrations &           0.938 $\pm $ 0.021 &     

In [14]:
print(results_latex_df.to_markdown())

|                                             | Accuracy                    | Silhouette Score            |
|:--------------------------------------------|:----------------------------|:----------------------------|
| ('Diversity Curves', 'Shortest Path')       | 0.992 $\pm $ 0.017          | 0.753 $\pm $ 0.019          |
| ('Embeddings', 'FEATHER')                   | \textbf{0.996 $\pm $ 0.012} | 0.688 $\pm $ 0.019          |
| ('Embeddings', 'NetLSD')                    | 0.979 $\pm $ 0.021          | 0.648 $\pm $ 0.028          |
| ('Embeddings', 'Spectral Zoo')              | 0.979 $\pm $ 0.021          | 0.675 $\pm $ 0.024          |
| ('Kernel', 'Kernel WL Optimal Assignment')  | \textbf{0.996 $\pm $ 0.012} | 0.089 $\pm $ 0.003          |
| ('Kernel', 'Kernel Shortest Path')          | 0.988 $\pm $ 0.019          | \textbf{0.772 $\pm $ 0.025} |
| ('TopER', 'TopER All Filtrations')          | 0.938 $\pm $ 0.021          | 0.398 $\pm $ 0.027          |
| ('Persistent Homology', 'P

## cluster nnodes 10 200 nsteps 5 ngraphs 1 mopcars 100 step comp card 5

In [2]:
# getting the right parameters
model_params = {
        'ER': {'ER_probs': [0.75], 
                'ER_nps': None,
        },
        'RPG':{
            'RPG_probs': [[0.9, 0.1]], 
            'RPG_props': [1/3, 1/3], 
        },
        'SBM': {
            'SBM_probs': [0.8, 0.05], 
            'SBM_blocks': [[1/4, 1/4, 1/4]]
            },
        'RG': {
            'RG_probs': [0.25]
        },
    }
    
models = list(model_params.keys())
all_configs = {}
for model in models:
    model_config = tuple(model_params[model][param] for param in model_params[model].keys())
    all_configs[model] = model_config
configuration = tuple(all_configs[model] for model in models)

# read dataframe
res_df = pd.read_csv(f'simulations/results_cluster/sim_distributions/nrange10_200nsteps_5ngraphs_1/max_comcard_100step_compute_card_5/res_df.csv', header=0, index_col=0)


# group methods
method_group = {}
for method in list(res_df.index):
   if 'toper' in method:
      method_group[method] = 'TopER'
   elif 'kernel' in method:
      method_group[method] = 'Kernel'
   elif 'embedding' in method or 'spectral_zoo' == method:
      method_group[method] = 'Embeddings'
   elif 'divcurves' in method:
      method_group[method] = 'Diversity Curves'
   elif 'div_function' in method or 'one_scale' in method:
      method_group[method] = 'Diversity Profile'
   elif 'persistent_homology' in method:
      method_group[method] = 'Persistent Homology'

      
res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
      'divcurves_shortest_path_distance': 'Shortest Path',
      'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
      'div_function_shortest_path_distance': 'Profile Shortest Path', 
      'graph_embedding_netLSD': 'NetLSD',
      'graph_embedding_graph2vec': 'Graph2Vec', 
      'graph_embedding_feathergraph': 'FEATHER',
      'kernel_shortest_path': 'Kernel Shortest Path', 
      'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
      'kernel_pyramid_match': 'Kernel Pyramid Match', 
      'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
      'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
      'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
      'kernel_weisfeiler_lehman': 'Kernel WL',
      'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
      'kernel_core_framework': 'Kernel Core Framework',
      'kernel_mds_core_framework': 'Kernel Core Framework MDS',
      'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
      'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
      'toper_degree_sub': 'TopER Degree', 
      'toper_popularity_sub': 'TopER Popularity', 
      'toper_forricci_sub': 'TopER Forricci',
      'toper_olricci_sub': 'TopER Olricci',
      'toper_closeness_sub': 'TopER Closeness', 
      'toper_all_sub': 'TopER All Filtrations',
      'spectral_zoo': 'Spectral Zoo', 
      'persistent_homology_degree': 'PH Degree',
      'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',
      'persistent_homology_forman_ricci_curvature': 'PH Fricci',
      'persistent_homology_laplacian': 'PH Laplacian', 
      'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
      'persistent_homology_alpha_complex': 'PH Alpha Complex',
}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)

# sort wrt to accuracy, in groups
df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)

scores = ['acc', 'silh', 'db', 'ch', 'ari_kmeans', 'ari_agglo']
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

res_copy = df.copy()

results_latex = {}

for score in scores:
   mean_col = f'{score}_mean'
   std_col = f'{score}_std'

   if mean_col in res_copy.columns and std_col in res_copy.columns:
      results_latex[score] = res_copy[mean_col].combine(
            res_copy[std_col],
            lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
      )

results_latex_df = pd.DataFrame.from_dict(results_latex)

# Bold the best mean for each score 
for score in scores:
   mean_col = f"{score}_mean"
   display_col =  score
   if mean_col in res_copy.columns:
      if not scores_asc[score]: 
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
      else:
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
      results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
            lambda x: f"\\textbf{{{x}}}"
      )
   else:
      print(f'{mean_col} not in dataframe?')

#final adjustments latex df 
results_latex_df.index.rename([None, 'Method'], inplace = True)
new_cols = {
   'acc': 'Accuracy', 
   'silh': 'Silhouette Score', 
   'db': 'Davies-Bouldin', 
   'ch': 'Calinski-Harabasz', 
   'ari_kmeans': 'ARI k-means', 
   'ari_agglo': 'ARI agglomerative'
}

results_latex_df = results_latex_df.rename(columns=new_cols)
results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

print(configuration)
print(results_latex_df.to_latex(multirow=False, escape=False))

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &              &                     Accuracy &             Silhouette Score &               Davies-Bouldin &              Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                &                              &                              \\
\midrule
Diversity Curves & Shortest Path &           0.862 $\pm $ 0.085 &           0.123 $\pm $ 0.038 &           1.560 $\pm $ 0.263 &            28.828 $\pm $ 6.096 &           0.245 $\pm $ 0.059 &           0.100 $\pm $ 0.072 \\
                    & Shortest Path PCA &           0.823 $\pm $ 0.088 &           0.124 $\pm $ 0.038 &           1.537 $\pm $ 0.259 &            29.068 $\pm $ 6.173 &           0.

In [3]:
df

silh_mean    db_mean  \
Group               Method                                                   
Diversity Curves    Shortest Path                      0.122643   1.560238   
                    Shortest Path PCA                  0.124382   1.537441   
Diversity Profile   Profile Shortest Path             -0.007681   4.406536   
Embeddings          NetLSD                            -0.147845   2.927892   
                    FEATHER                           -0.006370   3.724330   
                    Graph2Vec                         -0.100359   5.553694   
                    Spectral Zoo                      -0.153843   6.370146   
Kernel              Kernel Shortest Path               0.493663        NaN   
                    Kernel Shortest Path MDS           0.538877   1.197897   
                    Kernel WL                          0.449668        NaN   
                    Kernel WL MDS                      0.506936   1.608595   
                    Kernel Core Framework MDS          0.233213   2.002970   
                    Kernel Core Framework              0.223384        NaN   
                    Kernel WL Optimal Assignment      -0.002962        NaN   
                    Kernel WL Optimal Assignment MDS  -0.035523   8.558962   
                    Kernel Pyramid Match              -0.098511        NaN   
                    Kernel Neighbourhood Hash         -0.076432        NaN   
                    Kernel Neighbourhood Hash MDS     -0.098984  22.695147   
                    Kernel Pyramid Match MDS          -0.108718  24.144552   
TopER               TopER All Filtrations             -0.119748   4.425378   
                    TopER Olricci                     -0.141758  24.143467   
                    TopER Popularity                  -0.152042   8.020626   
                    TopER Forricci                    -0.122474  22.785002   
                    TopER Closeness                   -0.153304  13.554608   
                    TopER Degree                      -0.156294   7.843699   
Persistent Homology PH Alpha Complex                  -0.038711   3.656267   
                    PH Vietoris Rips                  -0.035755   5.368958   
                    PH Fricci                         -0.146607   1.852747   
                    PH Olricci                        -0.151973   1.867296   
                    PH Degree                         -0.219997   2.844157   
                    PH Laplacian                      -0.221977   3.121575   

                                                        ch_mean  silh_std  \
Group               Method                                                  
Diversity Curves    Shortest Path                     28.828447  0.037782   
                    Shortest Path PCA                 29.068493  0.038308   
Diversity Profile   Profile Shortest Path              7.411696  0.029897   
Embeddings          NetLSD                             3.819574  0.078724   
                    FEATHER                            9.354938  0.025027   
                    Graph2Vec                          9.805318  0.031984   
                    Spectral Zoo                       3.609380  0.039700   
Kernel              Kernel Shortest Path                    NaN  0.067069   
                    Kernel Shortest Path MDS          86.531319  0.063260   
                    Kernel WL                               NaN  0.057429   
                    Kernel WL MDS                     48.573952  0.062012   
                    Kernel Core Framework MDS         32.786450  0.039475   
                    Kernel Core Framework                   NaN  0.034666   
                    Kernel WL Optimal Assignment            NaN  0.002587   
                    Kernel WL Optimal Assignment MDS   3.214550  0.022109   
                    Kernel Pyramid Match                    NaN  0.017062   
                    Kernel Neighbourhood Hash               NaN  0.012392   
                

In [4]:
# getting the right parameters
model_params = {
        'ER': {'ER_probs': [0.75], 
                'ER_nps': None,
        },
        'RPG':{
            'RPG_probs': [[0.9, 0.1]], 
            'RPG_props': [1/3, 1/3], 
        },
        'SBM': {
            'SBM_probs': [0.8, 0.05], 
            'SBM_blocks': [[1/4, 1/4, 1/4]]
            },
        'RG': {
            'RG_probs': [0.25]
        },
    }
    
models = list(model_params.keys())
all_configs = {}
for model in models:
    model_config = tuple(model_params[model][param] for param in model_params[model].keys())
    all_configs[model] = model_config
configuration = tuple(all_configs[model] for model in models)

# read dataframe
res_df = pd.read_csv(f'/home/haeuserm/projects/diversity_curves/simulations/results_cluster/sim_distributions/nrange10_200nsteps_5ngraphs_1/max_comcard_150step_compute_card_5/res_df.csv', header=0, index_col=0)


# group methods
method_group = {}
for method in list(res_df.index):
   if 'toper' in method:
      method_group[method] = 'TopER'
   elif 'kernel' in method:
      method_group[method] = 'Kernel'
   elif 'embedding' in method or 'spectral_zoo' == method:
      method_group[method] = 'Embeddings'
   elif 'divcurves' in method:
      method_group[method] = 'Diversity Curves'
   elif 'div_function' in method or 'one_scale' in method:
      method_group[method] = 'Diversity Profile'
   elif 'persistent_homology' in method:
      method_group[method] = 'Persistent Homology'

      
res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
      'divcurves_shortest_path_distance': 'Shortest Path',
      'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
      'div_function_shortest_path_distance': 'Profile Shortest Path', 
      'graph_embedding_netLSD': 'NetLSD',
      'graph_embedding_graph2vec': 'Graph2Vec', 
      'graph_embedding_feathergraph': 'FEATHER',
      'kernel_shortest_path': 'Kernel Shortest Path', 
      'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
      'kernel_pyramid_match': 'Kernel Pyramid Match', 
      'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
      'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
      'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
      'kernel_weisfeiler_lehman': 'Kernel WL',
      'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
      'kernel_core_framework': 'Kernel Core Framework',
      'kernel_mds_core_framework': 'Kernel Core Framework MDS',
      'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
      'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
      'toper_degree_sub': 'TopER Degree', 
      'toper_popularity_sub': 'TopER Popularity', 
      'toper_forricci_sub': 'TopER Forricci',
      'toper_olricci_sub': 'TopER Olricci',
      'toper_closeness_sub': 'TopER Closeness', 
      'toper_all_sub': 'TopER All Filtrations',
      'spectral_zoo': 'Spectral Zoo', 
      'persistent_homology_degree': 'PH Degree',
      'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',
      'persistent_homology_forman_ricci_curvature': 'PH Fricci',
      'persistent_homology_laplacian': 'PH Laplacian', 
      'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
      'persistent_homology_alpha_complex': 'PH Alpha Complex',
}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)

# sort wrt to accuracy, in groups
df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)

scores = ['acc', 'silh', 'db', 'ch', 'ari_kmeans', 'ari_agglo']
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

res_copy = df.copy()

results_latex = {}

for score in scores:
   mean_col = f'{score}_mean'
   std_col = f'{score}_std'

   if mean_col in res_copy.columns and std_col in res_copy.columns:
      results_latex[score] = res_copy[mean_col].combine(
            res_copy[std_col],
            lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
      )

results_latex_df = pd.DataFrame.from_dict(results_latex)

# Bold the best mean for each score 
for score in scores:
   mean_col = f"{score}_mean"
   display_col =  score
   if mean_col in res_copy.columns:
      if not scores_asc[score]: 
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
      else:
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
      results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
            lambda x: f"\\textbf{{{x}}}"
      )
   else:
      print(f'{mean_col} not in dataframe?')

#final adjustments latex df 
results_latex_df.index.rename([None, 'Method'], inplace = True)
new_cols = {
   'acc': 'Accuracy', 
   'silh': 'Silhouette Score', 
   'db': 'Davies-Bouldin', 
   'ch': 'Calinski-Harabasz', 
   'ari_kmeans': 'ARI k-means', 
   'ari_agglo': 'ARI agglomerative'
}

results_latex_df = results_latex_df.rename(columns=new_cols)
results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

print(configuration)
print(results_latex_df.to_latex(multirow=False, escape=False))

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &              &                     Accuracy &             Silhouette Score &               Davies-Bouldin &              Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                &                              &                              \\
\midrule
Diversity Curves & Shortest Path &  \textbf{0.961 $\pm $ 0.044} &           0.263 $\pm $ 0.041 &           1.204 $\pm $ 0.181 &            45.340 $\pm $ 8.733 &           0.365 $\pm $ 0.096 &           0.215 $\pm $ 0.049 \\
                    & Shortest Path PCA &           0.954 $\pm $ 0.042 &           0.272 $\pm $ 0.042 &  \textbf{1.169 $\pm $ 0.178} &            46.079 $\pm $ 8.954 &           0.

In [5]:
df

silh_mean    db_mean  \
Group               Method                                                   
Diversity Curves    Shortest Path                      0.263179   1.204421   
                    Shortest Path PCA                  0.271984   1.168676   
Diversity Profile   Profile Shortest Path             -0.007681   4.406536   
Embeddings          NetLSD                            -0.147845   2.927892   
                    Graph2Vec                         -0.092545   5.245960   
                    FEATHER                           -0.006370   3.724330   
                    Spectral Zoo                      -0.153843   6.370146   
Kernel              Kernel Shortest Path               0.493663        NaN   
                    Kernel Shortest Path MDS           0.538877   1.197897   
                    Kernel WL                          0.449668        NaN   
                    Kernel WL MDS                      0.506936   1.608595   
                    Kernel Core Framework MDS          0.233213   2.002970   
                    Kernel Core Framework              0.223384        NaN   
                    Kernel WL Optimal Assignment      -0.002962        NaN   
                    Kernel WL Optimal Assignment MDS  -0.035523   8.558962   
                    Kernel Pyramid Match              -0.098511        NaN   
                    Kernel Neighbourhood Hash         -0.076432        NaN   
                    Kernel Neighbourhood Hash MDS     -0.098984  22.695147   
                    Kernel Pyramid Match MDS          -0.108718  24.144552   
TopER               TopER All Filtrations             -0.119748   4.425378   
                    TopER Olricci                     -0.141758  24.143467   
                    TopER Popularity                  -0.152042   8.020626   
                    TopER Forricci                    -0.122474  22.785002   
                    TopER Closeness                   -0.153304  13.554608   
                    TopER Degree                      -0.156294   7.843699   
Persistent Homology PH Alpha Complex                  -0.038711   3.656267   
                    PH Vietoris Rips                  -0.035755   5.368958   
                    PH Fricci                         -0.146607   1.852747   
                    PH Olricci                        -0.151973   1.867296   
                    PH Degree                         -0.219997   2.844157   
                    PH Laplacian                      -0.221977   3.121575   

                                                        ch_mean  silh_std  \
Group               Method                                                  
Diversity Curves    Shortest Path                     45.339749  0.040923   
                    Shortest Path PCA                 46.078576  0.042279   
Diversity Profile   Profile Shortest Path              7.411696  0.029897   
Embeddings          NetLSD                             3.819574  0.078724   
                    Graph2Vec                         10.289965  0.033470   
                    FEATHER                            9.354938  0.025027   
                    Spectral Zoo                       3.609380  0.039700   
Kernel              Kernel Shortest Path                    NaN  0.067069   
                    Kernel Shortest Path MDS          86.531319  0.063260   
                    Kernel WL                               NaN  0.057429   
                    Kernel WL MDS                     48.573952  0.062012   
                    Kernel Core Framework MDS         32.786450  0.039475   
                    Kernel Core Framework                   NaN  0.034666   
                    Kernel WL Optimal Assignment            NaN  0.002587   
                    Kernel WL Optimal Assignment MDS   3.214550  0.022109   
                    Kernel Pyramid Match                    NaN  0.017062   
                    Kernel Neighbourhood Hash               NaN  0.012392   
                

In [6]:
# getting the right parameters
model_params = {
        'ER': {'ER_probs': [0.75], 
                'ER_nps': None,
        },
        'RPG':{
            'RPG_probs': [[0.9, 0.1]], 
            'RPG_props': [1/3, 1/3], 
        },
        'SBM': {
            'SBM_probs': [0.8, 0.05], 
            'SBM_blocks': [[1/4, 1/4, 1/4]]
            },
        'RG': {
            'RG_probs': [0.25]
        },
    }
    
models = list(model_params.keys())
all_configs = {}
for model in models:
    model_config = tuple(model_params[model][param] for param in model_params[model].keys())
    all_configs[model] = model_config
configuration = tuple(all_configs[model] for model in models)

# read dataframe
res_df = pd.read_csv(f'simulations/results_cluster/sim_distributions/nrange10_200nsteps_10ngraphs_2/max_comcard_100step_compute_card_5/res_df.csv', header=0, index_col=0)


# group methods
method_group = {}
for method in list(res_df.index):
   if 'toper' in method:
      method_group[method] = 'TopER'
   elif 'kernel' in method:
      method_group[method] = 'Kernel'
   elif 'embedding' in method or 'spectral_zoo' == method:
      method_group[method] = 'Embeddings'
   elif 'divcurves' in method:
      method_group[method] = 'Diversity Curves'
   elif 'div_function' in method or 'one_scale' in method:
      method_group[method] = 'Diversity Profile'
   elif 'persistent_homology' in method:
      method_group[method] = 'Persistent Homology'

      
res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
      'divcurves_shortest_path_distance': 'Shortest Path',
      'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
      'div_function_shortest_path_distance': 'Profile Shortest Path', 
      'graph_embedding_netLSD': 'NetLSD',
      'graph_embedding_graph2vec': 'Graph2Vec', 
      'graph_embedding_feathergraph': 'FEATHER',
      'kernel_shortest_path': 'Kernel Shortest Path', 
      'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
      'kernel_pyramid_match': 'Kernel Pyramid Match', 
      'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
      'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
      'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
      'kernel_weisfeiler_lehman': 'Kernel WL',
      'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
      'kernel_core_framework': 'Kernel Core Framework',
      'kernel_mds_core_framework': 'Kernel Core Framework MDS',
      'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
      'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
      'toper_degree_sub': 'TopER Degree', 
      'toper_popularity_sub': 'TopER Popularity', 
      'toper_forricci_sub': 'TopER Forricci',
      'toper_olricci_sub': 'TopER Olricci',
      'toper_closeness_sub': 'TopER Closeness', 
      'toper_all_sub': 'TopER All Filtrations',
      'spectral_zoo': 'Spectral Zoo', 
      'persistent_homology_degree': 'PH Degree',
      'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',
      'persistent_homology_forman_ricci_curvature': 'PH Fricci',
      'persistent_homology_laplacian': 'PH Laplacian', 
      'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
      'persistent_homology_alpha_complex': 'PH Alpha Complex',
}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)

# sort wrt to accuracy, in groups
df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)

scores = ['acc', 'silh', 'db', 'ch', 'ari_kmeans', 'ari_agglo']
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

res_copy = df.copy()

results_latex = {}

for score in scores:
   mean_col = f'{score}_mean'
   std_col = f'{score}_std'

   if mean_col in res_copy.columns and std_col in res_copy.columns:
      results_latex[score] = res_copy[mean_col].combine(
            res_copy[std_col],
            lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
      )

results_latex_df = pd.DataFrame.from_dict(results_latex)

# Bold the best mean for each score 
for score in scores:
   mean_col = f"{score}_mean"
   display_col =  score
   if mean_col in res_copy.columns:
      if not scores_asc[score]: 
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
      else:
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
      results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
            lambda x: f"\\textbf{{{x}}}"
      )
   else:
      print(f'{mean_col} not in dataframe?')

#final adjustments latex df 
results_latex_df.index.rename([None, 'Method'], inplace = True)
new_cols = {
   'acc': 'Accuracy', 
   'silh': 'Silhouette Score', 
   'db': 'Davies-Bouldin', 
   'ch': 'Calinski-Harabasz', 
   'ari_kmeans': 'ARI k-means', 
   'ari_agglo': 'ARI agglomerative'
}

results_latex_df = results_latex_df.rename(columns=new_cols)
results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

print(configuration)
print(results_latex_df.to_latex(multirow=False, escape=False))

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &           &                     Accuracy &             Silhouette Score &               Davies-Bouldin &              Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                &                              &                              \\
\midrule
Diversity Curves & Shortest Path &           0.819 $\pm $ 0.065 &           0.133 $\pm $ 0.038 &           1.496 $\pm $ 0.225 &            30.477 $\pm $ 6.563 &           0.237 $\pm $ 0.054 &           0.101 $\pm $ 0.073 \\
                    & Shortest Path PCA &           0.800 $\pm $ 0.061 &           0.135 $\pm $ 0.039 &           1.472 $\pm $ 0.222 &            30.809 $\pm $ 6.681 &           0.241

In [7]:
# getting the right parameters
model_params = {
        'ER': {'ER_probs': [0.75], 
                'ER_nps': None,
        },
        'RPG':{
            'RPG_probs': [[0.9, 0.1]], 
            'RPG_props': [1/3, 1/3], 
        },
        'SBM': {
            'SBM_probs': [0.8, 0.05], 
            'SBM_blocks': [[1/4, 1/4, 1/4]]
            },
        'RG': {
            'RG_probs': [0.25]
        },
    }
    
models = list(model_params.keys())
all_configs = {}
for model in models:
    model_config = tuple(model_params[model][param] for param in model_params[model].keys())
    all_configs[model] = model_config
configuration = tuple(all_configs[model] for model in models)

# read dataframe
res_df = pd.read_csv(f'simulations/results_cluster/sim_distributions/nrange10_200nsteps_10ngraphs_2/max_comcard_150step_compute_card_5/res_df.csv', header=0, index_col=0)


# group methods
method_group = {}
for method in list(res_df.index):
   if 'toper' in method:
      method_group[method] = 'TopER'
   elif 'kernel' in method:
      method_group[method] = 'Kernel'
   elif 'embedding' in method or 'spectral_zoo' == method:
      method_group[method] = 'Embeddings'
   elif 'divcurves' in method:
      method_group[method] = 'Diversity Curves'
   elif 'div_function' in method or 'one_scale' in method:
      method_group[method] = 'Diversity Profile'
   elif 'persistent_homology' in method:
      method_group[method] = 'Persistent Homology'

      
res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
      'divcurves_shortest_path_distance': 'Shortest Path',
      'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
      'div_function_shortest_path_distance': 'Profile Shortest Path', 
      'graph_embedding_netLSD': 'NetLSD',
      'graph_embedding_graph2vec': 'Graph2Vec', 
      'graph_embedding_feathergraph': 'FEATHER',
      'kernel_shortest_path': 'Kernel Shortest Path', 
      'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
      'kernel_pyramid_match': 'Kernel Pyramid Match', 
      'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
      'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
      'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
      'kernel_weisfeiler_lehman': 'Kernel WL',
      'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
      'kernel_core_framework': 'Kernel Core Framework',
      'kernel_mds_core_framework': 'Kernel Core Framework MDS',
      'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
      'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
      'toper_degree_sub': 'TopER Degree', 
      'toper_popularity_sub': 'TopER Popularity', 
      'toper_forricci_sub': 'TopER Forricci',
      'toper_olricci_sub': 'TopER Olricci',
      'toper_closeness_sub': 'TopER Closeness', 
      'toper_all_sub': 'TopER All Filtrations',
      'spectral_zoo': 'Spectral Zoo', 
      'persistent_homology_degree': 'PH Degree',
      'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',
      'persistent_homology_forman_ricci_curvature': 'PH Fricci',
      'persistent_homology_laplacian': 'PH Laplacian', 
      'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
      'persistent_homology_alpha_complex': 'PH Alpha Complex',
}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)

# sort wrt to accuracy, in groups
df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)

scores = ['acc', 'silh', 'db', 'ch', 'ari_kmeans', 'ari_agglo']
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

res_copy = df.copy()

results_latex = {}

for score in scores:
   mean_col = f'{score}_mean'
   std_col = f'{score}_std'

   if mean_col in res_copy.columns and std_col in res_copy.columns:
      results_latex[score] = res_copy[mean_col].combine(
            res_copy[std_col],
            lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
      )

results_latex_df = pd.DataFrame.from_dict(results_latex)

# Bold the best mean for each score 
for score in scores:
   mean_col = f"{score}_mean"
   display_col =  score
   if mean_col in res_copy.columns:
      if not scores_asc[score]: 
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
      else:
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
      results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
            lambda x: f"\\textbf{{{x}}}"
      )
   else:
      print(f'{mean_col} not in dataframe?')

#final adjustments latex df 
results_latex_df.index.rename([None, 'Method'], inplace = True)
new_cols = {
   'acc': 'Accuracy', 
   'silh': 'Silhouette Score', 
   'db': 'Davies-Bouldin', 
   'ch': 'Calinski-Harabasz', 
   'ari_kmeans': 'ARI k-means', 
   'ari_agglo': 'ARI agglomerative'
}

results_latex_df = results_latex_df.rename(columns=new_cols)
results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

print(configuration)
print(results_latex_df.to_latex(multirow=False, escape=False))

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &           &                     Accuracy &             Silhouette Score &               Davies-Bouldin &              Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                &                              &                              \\
\midrule
Diversity Curves & Shortest Path PCA &           0.925 $\pm $ 0.067 &           0.288 $\pm $ 0.042 &           1.114 $\pm $ 0.156 &            49.602 $\pm $ 9.963 &           0.387 $\pm $ 0.097 &           0.213 $\pm $ 0.053 \\
                    & Shortest Path &           0.919 $\pm $ 0.063 &           0.278 $\pm $ 0.041 &           1.152 $\pm $ 0.159 &            48.645 $\pm $ 9.661 &           0.383

In [8]:
df

silh_mean    db_mean  \
Group               Method                                                   
Diversity Curves    Shortest Path PCA                  0.287745   1.114340   
                    Shortest Path                      0.278256   1.151888   
Diversity Profile   Profile Shortest Path             -0.013054   4.642434   
Embeddings          NetLSD                            -0.164632   3.011523   
                    FEATHER                           -0.007455   3.732177   
                    Spectral Zoo                      -0.171875   6.436683   
                    Graph2Vec                         -0.111183   5.295187   
Kernel              Kernel Shortest Path               0.503814        NaN   
                    Kernel Shortest Path MDS           0.544257   1.083806   
                    Kernel WL                          0.447893        NaN   
                    Kernel WL MDS                      0.498486   1.525336   
                    Kernel Core Framework              0.222193        NaN   
                    Kernel Core Framework MDS          0.211811   2.067162   
                    Kernel WL Optimal Assignment      -0.003208        NaN   
                    Kernel WL Optimal Assignment MDS  -0.061863   9.819284   
                    Kernel Pyramid Match              -0.098518        NaN   
                    Kernel Pyramid Match MDS          -0.108490  22.464600   
                    Kernel Neighbourhood Hash         -0.077414        NaN   
                    Kernel Neighbourhood Hash MDS     -0.098539  20.395316   
TopER               TopER All Filtrations             -0.119292   4.134429   
                    TopER Forricci                    -0.115210  25.814540   
                    TopER Olricci                     -0.146250  11.273777   
                    TopER Popularity                  -0.153105   9.156261   
                    TopER Closeness                   -0.154993  12.581688   
                    TopER Degree                      -0.159882   7.198862   
Persistent Homology PH Vietoris Rips                  -0.038365   5.330559   
                    PH Alpha Complex                  -0.037441   3.673469   
                    PH Fricci                         -0.152436   1.863898   
                    PH Olricci                        -0.159021   1.914394   
                    PH Laplacian                      -0.222741   3.185298   
                    PH Degree                         -0.223620   2.937187   

                                                        ch_mean  silh_std  \
Group               Method                                                  
Diversity Curves    Shortest Path PCA                 49.602396  0.042099   
                    Shortest Path                     48.645394  0.040785   
Diversity Profile   Profile Shortest Path              6.861194  0.029650   
Embeddings          NetLSD                             3.281238  0.084394   
                    FEATHER                            8.972925  0.025044   
                    Spectral Zoo                       2.942784  0.041054   
                    Graph2Vec                          9.778681  0.030629   
Kernel              Kernel Shortest Path                    NaN  0.068367   
                    Kernel Shortest Path MDS          93.760354  0.063753   
                    Kernel WL                               NaN  0.058116   
                    Kernel WL MDS                     49.223696  0.063496   
                    Kernel Core Framework                   NaN  0.035081   
                    Kernel Core Framework MDS         29.064760  0.040933   
                    Kernel WL Optimal Assignment            NaN  0.002612   
                    Kernel WL Optimal Assignment MDS   2.153089  0.013575   
                    Kernel Pyramid Match                    NaN  0.017310   
                    Kernel Pyramid Match MDS           0.518823  0.018083   
                

In [11]:
# getting the right parameters
model_params = {
        'ER': {'ER_probs': [0.75], 
                'ER_nps': None,
        },
        'RPG':{
            'RPG_probs': [[0.9, 0.1]], 
            'RPG_props': [1/3, 1/3], 
        },
        'SBM': {
            'SBM_probs': [0.8, 0.05], 
            'SBM_blocks': [[1/4, 1/4, 1/4]]
            },
        'RG': {
            'RG_probs': [0.25]
        },
    }
    
models = list(model_params.keys())
all_configs = {}
for model in models:
    model_config = tuple(model_params[model][param] for param in model_params[model].keys())
    all_configs[model] = model_config
configuration = tuple(all_configs[model] for model in models)

# read dataframe
res_df = pd.read_csv(f'simulations/results_cluster/sim_distributions/nrange10_300nsteps_10ngraphs_1/max_comcard_100step_compute_card_5/res_df.csv', header=0, index_col=0)


# group methods
method_group = {}
for method in list(res_df.index):
   if 'toper' in method:
      method_group[method] = 'TopER'
   elif 'kernel' in method:
      method_group[method] = 'Kernel'
   elif 'embedding' in method or 'spectral_zoo' == method:
      method_group[method] = 'Embeddings'
   elif 'divcurves' in method:
      method_group[method] = 'Diversity Curves'
   elif 'div_function' in method or 'one_scale' in method:
      method_group[method] = 'Diversity Profile'
   elif 'persistent_homology' in method:
      method_group[method] = 'Persistent Homology'

      
res_df['group'] = [method_group[method] for method in list(res_df.index)]

#rename rows, only relevant ones, rest will be dropped
new_names = {
      'divcurves_shortest_path_distance': 'Shortest Path',
      'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
      'div_function_shortest_path_distance': 'Profile Shortest Path', 
      'graph_embedding_netLSD': 'NetLSD',
      'graph_embedding_graph2vec': 'Graph2Vec', 
      'graph_embedding_feathergraph': 'FEATHER',
      'kernel_shortest_path': 'Kernel Shortest Path', 
      'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
      'kernel_pyramid_match': 'Kernel Pyramid Match', 
      'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
      'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
      'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
      'kernel_weisfeiler_lehman': 'Kernel WL',
      'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
      'kernel_core_framework': 'Kernel Core Framework',
      'kernel_mds_core_framework': 'Kernel Core Framework MDS',
      'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
      'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
      'toper_degree_sub': 'TopER Degree', 
      'toper_popularity_sub': 'TopER Popularity', 
      'toper_forricci_sub': 'TopER Forricci',
      'toper_olricci_sub': 'TopER Olricci',
      'toper_closeness_sub': 'TopER Closeness', 
      'toper_all_sub': 'TopER All Filtrations',
      'spectral_zoo': 'Spectral Zoo', 
      'persistent_homology_degree': 'PH Degree',
      'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',
      'persistent_homology_forman_ricci_curvature': 'PH Fricci',
      'persistent_homology_laplacian': 'PH Laplacian', 
      # 'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
      # 'persistent_homology_alpha_complex': 'PH Alpha Complex',
}

df = res_df.rename(new_names)
df = df.loc[list(new_names.values()),:]

# add group to multiindex
df = df.set_index('group', append = True).swaplevel(0,1)
df.index.rename(['Group', 'Method'], inplace = True)

# sort wrt to accuracy, in groups
df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)

scores = ['acc', 'silh', 'db', 'ch', 'ari_kmeans', 'ari_agglo']
scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

res_copy = df.copy()

results_latex = {}

for score in scores:
   mean_col = f'{score}_mean'
   std_col = f'{score}_std'

   if mean_col in res_copy.columns and std_col in res_copy.columns:
      results_latex[score] = res_copy[mean_col].combine(
            res_copy[std_col],
            lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
      )

results_latex_df = pd.DataFrame.from_dict(results_latex)

# Bold the best mean for each score 
for score in scores:
   mean_col = f"{score}_mean"
   display_col =  score
   if mean_col in res_copy.columns:
      if not scores_asc[score]: 
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
      else:
            best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
      results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
            lambda x: f"\\textbf{{{x}}}"
      )
   else:
      print(f'{mean_col} not in dataframe?')

#final adjustments latex df 
results_latex_df.index.rename([None, 'Method'], inplace = True)
new_cols = {
   'acc': 'Accuracy', 
   'silh': 'Silhouette Score', 
   'db': 'Davies-Bouldin', 
   'ch': 'Calinski-Harabasz', 
   'ari_kmeans': 'ARI k-means', 
   'ari_agglo': 'ARI agglomerative'
}

results_latex_df = results_latex_df.rename(columns=new_cols)
results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

print(configuration)
print(results_latex_df.to_latex(multirow=False, escape=False))

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &              &                     Accuracy &             Silhouette Score &               Davies-Bouldin &                Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                  &                              &                              \\
\midrule
Diversity Curves & Shortest Path &           0.697 $\pm $ 0.073 &           0.006 $\pm $ 0.057 &           2.209 $\pm $ 0.750 &              13.613 $\pm $ 4.167 &           0.197 $\pm $ 0.044 &           0.050 $\pm $ 0.056 \\
                    & Shortest Path PCA &           0.635 $\pm $ 0.110 &           0.002 $\pm $ 0.058 &           2.194 $\pm $ 0.751 &              13.705 $\pm $ 4.220 &     

In [12]:
df

silh_mean    db_mean  \
Group               Method                                                   
Diversity Curves    Shortest Path                      0.005562   2.209070   
                    Shortest Path PCA                  0.001915   2.193762   
Diversity Profile   Profile Shortest Path              0.003917   3.869578   
Embeddings          NetLSD                            -0.168835   2.927051   
                    Graph2Vec                         -0.061267   5.577141   
                    FEATHER                           -0.016042   3.769315   
                    Spectral Zoo                      -0.192085   6.978964   
Kernel              Kernel Shortest Path MDS           0.651668   0.906047   
                    Kernel WL MDS                      0.614541   1.182471   
                    Kernel Shortest Path               0.576934        NaN   
                    Kernel WL                          0.516410        NaN   
                    Kernel Core Framework MDS          0.260273   1.933730   
                    Kernel Core Framework              0.230288        NaN   
                    Kernel WL Optimal Assignment      -0.005689        NaN   
                    Kernel WL Optimal Assignment MDS  -0.061844  11.751549   
                    Kernel Pyramid Match              -0.120706        NaN   
                    Kernel Neighbourhood Hash         -0.096796        NaN   
                    Kernel Neighbourhood Hash MDS     -0.118715  17.328950   
                    Kernel Pyramid Match MDS          -0.129457  18.559975   
TopER               TopER All Filtrations             -0.111189   5.110255   
                    TopER Olricci                     -0.156374  15.184267   
                    TopER Degree                      -0.164223   9.350040   
                    TopER Closeness                   -0.142314   9.921726   
                    TopER Forricci                    -0.162712  11.289302   
                    TopER Popularity                  -0.162354  11.427944   
Persistent Homology PH Olricci                        -0.145846   1.900683   
                    PH Fricci                         -0.130599   1.636139   
                    PH Degree                         -0.212770   2.780347   
                    PH Laplacian                      -0.217752   2.850069   

                                                         ch_mean  silh_std  \
Group               Method                                                   
Diversity Curves    Shortest Path                      13.612615  0.057064   
                    Shortest Path PCA                  13.705269  0.057972   
Diversity Profile   Profile Shortest Path               7.373598  0.038000   
Embeddings          NetLSD                              2.710113  0.123331   
                    Graph2Vec                           7.457139  0.049885   
                    FEATHER                             6.316208  0.027009   
                    Spectral Zoo                        2.067353  0.059617   
Kernel              Kernel Shortest Path MDS          141.397827  0.075564   
                    Kernel WL MDS                      89.773099  0.078956   
                    Kernel Shortest Path                     NaN  0.100075   
                    Kernel WL                                NaN  0.082107   
                    Kernel Core Framework MDS          27.345898  0.039358   
                    Kernel Core Framework                    NaN  0.034896   
                    Kernel WL Optimal Assignment             NaN  0.002982   
                    Kernel WL Optimal Assignment MDS    1.103772  0.016618   
                    Kernel Pyramid Match                     NaN  0.021026   
                    Kernel Neighbourhood Hash                NaN  0.015609   
                    Kernel Neighbourhood Hash MDS       0.543315  0.017461   
                    Kernel Pyramid Match MDS            0.557773  0.021742 

## function for table gen

In [2]:
def generate_table(dir_path):
    # getting the right parameters
   model_params = {
         'ER': {'ER_probs': [0.75], 
                  'ER_nps': None,
         },
         'RPG':{
               'RPG_probs': [[0.9, 0.1]], 
               'RPG_props': [1/3, 1/3], 
         },
         'SBM': {
               'SBM_probs': [0.8, 0.05], 
               'SBM_blocks': [[1/4, 1/4, 1/4]]
               },
         'RG': {
               'RG_probs': [0.25]
         },
      }
      
   models = list(model_params.keys())
   all_configs = {}
   for model in models:
      model_config = tuple(model_params[model][param] for param in model_params[model].keys())
      all_configs[model] = model_config
   configuration = tuple(all_configs[model] for model in models)

   # read dataframe
   res_df = pd.read_csv(dir_path, header=0, index_col=0)


   # group methods
   method_group = {}
   for method in list(res_df.index):
      if 'toper' in method:
         method_group[method] = 'TopER'
      elif 'kernel' in method:
         method_group[method] = 'Kernel'
      elif 'embedding' in method or 'spectral_zoo' == method:
         method_group[method] = 'Embeddings'
      elif 'divcurves' in method:
         method_group[method] = 'Diversity Curves'
      elif 'div_function' in method or 'one_scale' in method:
         method_group[method] = 'Diversity Profile'
      elif 'persistent_homology' in method:
         method_group[method] = 'Persistent Homology'

         
   res_df['group'] = [method_group[method] for method in list(res_df.index)]

   #rename rows, only relevant ones, rest will be dropped
   new_names = {
         'divcurves_shortest_path_distance': 'Shortest Path',
         'divcurves_pca_shortest_path_distance': 'Shortest Path PCA',
         'div_function_shortest_path_distance': 'Profile Shortest Path', 
         'graph_embedding_netLSD': 'NetLSD',
         'graph_embedding_graph2vec': 'Graph2Vec', 
         'graph_embedding_feathergraph': 'FEATHER',
         'kernel_shortest_path': 'Kernel Shortest Path', 
         'kernel_mds_shortest_path': 'Kernel Shortest Path MDS',
         'kernel_pyramid_match': 'Kernel Pyramid Match', 
         'kernel_mds_pyramid_match': 'Kernel Pyramid Match MDS',
         'kernel_neighborhood_hash': 'Kernel Neighbourhood Hash', 
         'kernel_mds_neighborhood_hash': 'Kernel Neighbourhood Hash MDS',
         'kernel_weisfeiler_lehman': 'Kernel WL',
         'kernel_mds_weisfeiler_lehman': 'Kernel WL MDS', 
         'kernel_core_framework': 'Kernel Core Framework',
         'kernel_mds_core_framework': 'Kernel Core Framework MDS',
         'kernel_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment',
         'kernel_mds_weisfeiler_lehman_optimal_assignment': 'Kernel WL Optimal Assignment MDS', 
         'toper_degree_sub': 'TopER Degree', 
         'toper_popularity_sub': 'TopER Popularity', 
         'toper_forricci_sub': 'TopER Forricci',
         'toper_olricci_sub': 'TopER Olricci',
         'toper_closeness_sub': 'TopER Closeness', 
         'toper_all_sub': 'TopER All Filtrations',
         'spectral_zoo': 'Spectral Zoo', 
         'persistent_homology_degree': 'PH Degree',
         'persistent_homology_ollivier_ricci_curvature': 'PH Olricci',
         'persistent_homology_forman_ricci_curvature': 'PH Fricci',
         'persistent_homology_laplacian': 'PH Laplacian', 
         # 'persistent_homology_vietoris_rips': 'PH Vietoris Rips',
         # 'persistent_homology_alpha_complex': 'PH Alpha Complex',
   }

   df = res_df.rename(new_names)
   df = df.loc[list(new_names.values()),:]

   # add group to multiindex
   df = df.set_index('group', append = True).swaplevel(0,1)
   df.index.rename(['Group', 'Method'], inplace = True)

   # sort wrt to accuracy, in groups
   df = df.groupby(level=0, sort=False).apply(lambda x: x.sort_values('acc_mean', ascending=False)).droplevel(0)

   scores = ['acc', 'silh', 'db', 'ch', 'ari_kmeans', 'ari_agglo']
   scores_asc = {'acc': False, 'silh': False, 'ari_agglo': False, 'db': True,'ch': False, 'ari_kmeans': False}

   res_copy = df.copy()

   results_latex = {}

   for score in scores:
      mean_col = f'{score}_mean'
      std_col = f'{score}_std'

      if mean_col in res_copy.columns and std_col in res_copy.columns:
         results_latex[score] = res_copy[mean_col].combine(
               res_copy[std_col],
               lambda m, s: '{:.3f}'.format(round(m, 3)) + ' $\pm $ ' + '{:.3f}'.format(round(s, 3))
         )

   results_latex_df = pd.DataFrame.from_dict(results_latex)

   # Bold the best mean for each score 
   for score in scores:
      mean_col = f"{score}_mean"
      display_col =  score
      if mean_col in res_copy.columns:
         if not scores_asc[score]: 
               best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].max(),3)
         else:
               best_mask = round(res_copy[mean_col],3) == round(res_copy[mean_col].min(),3)
         results_latex_df.loc[best_mask, display_col] = results_latex_df.loc[best_mask, display_col].apply(
               lambda x: f"\\textbf{{{x}}}"
         )
      else:
         print(f'{mean_col} not in dataframe?')

   #final adjustments latex df 
   results_latex_df.index.rename([None, 'Method'], inplace = True)
   new_cols = {
      'acc': 'Accuracy', 
      'silh': 'Silhouette Score', 
      'db': 'Davies-Bouldin', 
      'ch': 'Calinski-Harabasz', 
      'ari_kmeans': 'ARI k-means', 
      'ari_agglo': 'ARI agglomerative'
   }

   results_latex_df = results_latex_df.rename(columns=new_cols)
   results_latex_df.replace(to_replace='nan $\\pm $ nan', value='-', inplace=True)

   print(configuration)
   print(results_latex_df.to_latex(multirow=False, escape=False))
   return df

In [19]:
df = generate_table('simulations/results_cluster/sim_distributions/nrange10_200nsteps_5ngraphs_1_test/max_comcard_200step_compute_card_4/res_df.csv')

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &              &                     Accuracy &             Silhouette Score &               Davies-Bouldin &              Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                &                              &                              \\
\midrule
Diversity Curves & Shortest Path &  \textbf{0.968 $\pm $ 0.044} &           0.355 $\pm $ 0.043 &           1.046 $\pm $ 0.154 &           60.415 $\pm $ 10.824 &           0.478 $\pm $ 0.127 &           0.241 $\pm $ 0.034 \\
                    & Shortest Path PCA &  \textbf{0.968 $\pm $ 0.044} &           0.369 $\pm $ 0.045 &  \textbf{1.011 $\pm $ 0.154} &           61.810 $\pm $ 11.212 &           0.

In [20]:
df

silh_mean    db_mean  \
Group               Method                                                   
Diversity Curves    Shortest Path                      0.355203   1.045806   
                    Shortest Path PCA                  0.369035   1.010637   
Diversity Profile   Profile Shortest Path             -0.007681   4.406536   
Embeddings          NetLSD                            -0.147845   2.927892   
                    FEATHER                           -0.006370   3.724330   
                    Graph2Vec                         -0.095156   4.742061   
                    Spectral Zoo                      -0.153843   6.370146   
Kernel              Kernel Shortest Path               0.493663        NaN   
                    Kernel Shortest Path MDS           0.538877   1.197897   
                    Kernel WL                          0.449668        NaN   
                    Kernel WL MDS                      0.506936   1.608595   
                    Kernel Core Framework MDS          0.233213   2.002970   
                    Kernel Core Framework              0.223384        NaN   
                    Kernel WL Optimal Assignment      -0.002962        NaN   
                    Kernel WL Optimal Assignment MDS  -0.035523   8.558962   
                    Kernel Pyramid Match              -0.098511        NaN   
                    Kernel Neighbourhood Hash         -0.076432        NaN   
                    Kernel Neighbourhood Hash MDS     -0.098984  22.695147   
                    Kernel Pyramid Match MDS          -0.108718  24.144552   
TopER               TopER All Filtrations             -0.119748   4.425378   
                    TopER Olricci                     -0.141758  24.143467   
                    TopER Popularity                  -0.152042   8.020626   
                    TopER Forricci                    -0.122474  22.785002   
                    TopER Closeness                   -0.153304  13.554608   
                    TopER Degree                      -0.156294   7.843699   
Persistent Homology PH Fricci                         -0.146607   1.852747   
                    PH Olricci                        -0.151973   1.867296   
                    PH Degree                         -0.219997   2.844157   
                    PH Laplacian                      -0.221977   3.121575   

                                                        ch_mean  silh_std  \
Group               Method                                                  
Diversity Curves    Shortest Path                     60.415258  0.043359   
                    Shortest Path PCA                 61.810256  0.045098   
Diversity Profile   Profile Shortest Path              7.411696  0.029897   
Embeddings          NetLSD                             3.819574  0.078724   
                    FEATHER                            9.354938  0.025027   
                    Graph2Vec                          9.973433  0.034618   
                    Spectral Zoo                       3.609380  0.039700   
Kernel              Kernel Shortest Path                    NaN  0.067069   
                    Kernel Shortest Path MDS          86.531319  0.063260   
                    Kernel WL                               NaN  0.057429   
                    Kernel WL MDS                     48.573952  0.062012   
                    Kernel Core Framework MDS         32.786450  0.039475   
                    Kernel Core Framework                   NaN  0.034666   
                    Kernel WL Optimal Assignment            NaN  0.002587   
                    Kernel WL Optimal Assignment MDS   3.214550  0.022109   
                    Kernel Pyramid Match                    NaN  0.017062   
                    Kernel Neighbourhood Hash               NaN  0.012392   
                    Kernel Neighbourhood Hash MDS      0.523809  0.013950   
                    Kernel Pyramid Match MDS           0.517605  0.017851   
TopER             

In [21]:
df = generate_table('simulations/results_cluster/sim_distributions/nrange10_200nsteps_5ngraphs_1_test/max_comcard_200step_compute_card_3/res_df.csv')

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &              &                     Accuracy &             Silhouette Score &               Davies-Bouldin &              Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                &                              &                              \\
\midrule
Diversity Curves & Shortest Path &  \textbf{0.968 $\pm $ 0.044} &           0.354 $\pm $ 0.043 &           1.047 $\pm $ 0.154 &           60.248 $\pm $ 10.795 &           0.479 $\pm $ 0.124 &           0.241 $\pm $ 0.034 \\
                    & Shortest Path PCA &  \textbf{0.968 $\pm $ 0.044} &           0.368 $\pm $ 0.045 &  \textbf{1.012 $\pm $ 0.154} &           61.636 $\pm $ 11.182 &           0.

In [22]:
df

silh_mean    db_mean  \
Group               Method                                                   
Diversity Curves    Shortest Path                      0.354368   1.047090   
                    Shortest Path PCA                  0.368171   1.011850   
Diversity Profile   Profile Shortest Path             -0.007681   4.406536   
Embeddings          NetLSD                            -0.147845   2.927892   
                    FEATHER                           -0.006370   3.724330   
                    Graph2Vec                         -0.109932   5.181878   
                    Spectral Zoo                      -0.153843   6.370146   
Kernel              Kernel Shortest Path               0.493663        NaN   
                    Kernel Shortest Path MDS           0.538877   1.197897   
                    Kernel WL                          0.449668        NaN   
                    Kernel WL MDS                      0.506936   1.608595   
                    Kernel Core Framework MDS          0.233213   2.002970   
                    Kernel Core Framework              0.223384        NaN   
                    Kernel WL Optimal Assignment      -0.002962        NaN   
                    Kernel WL Optimal Assignment MDS  -0.035523   8.558962   
                    Kernel Pyramid Match              -0.098511        NaN   
                    Kernel Neighbourhood Hash         -0.076432        NaN   
                    Kernel Neighbourhood Hash MDS     -0.098984  22.695147   
                    Kernel Pyramid Match MDS          -0.108718  24.144552   
TopER               TopER All Filtrations             -0.119748   4.425378   
                    TopER Olricci                     -0.141758  24.143467   
                    TopER Popularity                  -0.152042   8.020626   
                    TopER Forricci                    -0.122474  22.785002   
                    TopER Closeness                   -0.153304  13.554608   
                    TopER Degree                      -0.156294   7.843699   
Persistent Homology PH Fricci                         -0.146607   1.852747   
                    PH Olricci                        -0.151973   1.867296   
                    PH Degree                         -0.219997   2.844157   
                    PH Laplacian                      -0.221977   3.121575   

                                                        ch_mean  silh_std  \
Group               Method                                                  
Diversity Curves    Shortest Path                     60.248203  0.043340   
                    Shortest Path PCA                 61.636275  0.045071   
Diversity Profile   Profile Shortest Path              7.411696  0.029897   
Embeddings          NetLSD                             3.819574  0.078724   
                    FEATHER                            9.354938  0.025027   
                    Graph2Vec                          9.784277  0.031630   
                    Spectral Zoo                       3.609380  0.039700   
Kernel              Kernel Shortest Path                    NaN  0.067069   
                    Kernel Shortest Path MDS          86.531319  0.063260   
                    Kernel WL                               NaN  0.057429   
                    Kernel WL MDS                     48.573952  0.062012   
                    Kernel Core Framework MDS         32.786450  0.039475   
                    Kernel Core Framework                   NaN  0.034666   
                    Kernel WL Optimal Assignment            NaN  0.002587   
                    Kernel WL Optimal Assignment MDS   3.214550  0.022109   
                    Kernel Pyramid Match                    NaN  0.017062   
                    Kernel Neighbourhood Hash               NaN  0.012392   
                    Kernel Neighbourhood Hash MDS      0.523809  0.013950   
                    Kernel Pyramid Match MDS           0.517605  0.017851   
TopER             

In [23]:
df = generate_table('simulations/results_cluster/sim_distributions/nrange10_200nsteps_5ngraphs_1_test/max_comcard_200step_compute_card_2/res_df.csv')

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &              &                     Accuracy &             Silhouette Score &               Davies-Bouldin &              Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                &                              &                              \\
\midrule
Diversity Curves & Shortest Path &  \textbf{0.968 $\pm $ 0.044} &           0.357 $\pm $ 0.043 &           1.043 $\pm $ 0.154 &           60.706 $\pm $ 10.861 &           0.475 $\pm $ 0.129 &           0.241 $\pm $ 0.035 \\
                    & Shortest Path PCA &  \textbf{0.968 $\pm $ 0.044} &           0.371 $\pm $ 0.045 &  \textbf{1.008 $\pm $ 0.154} &           62.111 $\pm $ 11.252 &           0.

In [24]:
df

silh_mean    db_mean  \
Group               Method                                                   
Diversity Curves    Shortest Path                      0.356687   1.043369   
                    Shortest Path PCA                  0.370578   1.008308   
Diversity Profile   Profile Shortest Path             -0.007681   4.406536   
Embeddings          NetLSD                            -0.147845   2.927892   
                    FEATHER                           -0.006370   3.724330   
                    Graph2Vec                         -0.100741   5.041740   
                    Spectral Zoo                      -0.153843   6.370146   
Kernel              Kernel Shortest Path               0.493663        NaN   
                    Kernel Shortest Path MDS           0.538877   1.197897   
                    Kernel WL                          0.449668        NaN   
                    Kernel WL MDS                      0.506936   1.608595   
                    Kernel Core Framework MDS          0.233213   2.002970   
                    Kernel Core Framework              0.223384        NaN   
                    Kernel WL Optimal Assignment      -0.002962        NaN   
                    Kernel WL Optimal Assignment MDS  -0.035523   8.558962   
                    Kernel Pyramid Match              -0.098511        NaN   
                    Kernel Neighbourhood Hash         -0.076432        NaN   
                    Kernel Neighbourhood Hash MDS     -0.098984  22.695147   
                    Kernel Pyramid Match MDS          -0.108718  24.144552   
TopER               TopER All Filtrations             -0.119748   4.425378   
                    TopER Olricci                     -0.141758  24.143467   
                    TopER Popularity                  -0.152042   8.020626   
                    TopER Forricci                    -0.122474  22.785002   
                    TopER Closeness                   -0.153304  13.554608   
                    TopER Degree                      -0.156294   7.843699   
Persistent Homology PH Fricci                         -0.146607   1.852747   
                    PH Olricci                        -0.151973   1.867296   
                    PH Degree                         -0.219997   2.844157   
                    PH Laplacian                      -0.221977   3.121575   

                                                        ch_mean  silh_std  \
Group               Method                                                  
Diversity Curves    Shortest Path                     60.706088  0.043409   
                    Shortest Path PCA                 62.110907  0.045147   
Diversity Profile   Profile Shortest Path              7.411696  0.029897   
Embeddings          NetLSD                             3.819574  0.078724   
                    FEATHER                            9.354938  0.025027   
                    Graph2Vec                          9.888322  0.033467   
                    Spectral Zoo                       3.609380  0.039700   
Kernel              Kernel Shortest Path                    NaN  0.067069   
                    Kernel Shortest Path MDS          86.531319  0.063260   
                    Kernel WL                               NaN  0.057429   
                    Kernel WL MDS                     48.573952  0.062012   
                    Kernel Core Framework MDS         32.786450  0.039475   
                    Kernel Core Framework                   NaN  0.034666   
                    Kernel WL Optimal Assignment            NaN  0.002587   
                    Kernel WL Optimal Assignment MDS   3.214550  0.022109   
                    Kernel Pyramid Match                    NaN  0.017062   
                    Kernel Neighbourhood Hash               NaN  0.012392   
                    Kernel Neighbourhood Hash MDS      0.523809  0.013950   
                    Kernel Pyramid Match MDS           0.517605  0.017851   
TopER             

In [3]:
df = generate_table('simulations/results_cluster/sim_distributions/nrange10_30_100_130_200_230_300_330nsteps_1ngraphs_1/max_comcard_150step_compute_card_5/res_df.csv')

KeyError: "['Kernel Core Framework', 'Kernel Core Framework MDS', 'Kernel WL Optimal Assignment', 'Kernel WL Optimal Assignment MDS', 'TopER Degree', 'TopER Popularity', 'TopER Forricci', 'TopER Olricci', 'TopER Closeness', 'TopER All Filtrations', 'Spectral Zoo', 'PH Degree', 'PH Olricci', 'PH Fricci', 'PH Laplacian'] not in index"

In [4]:
df = generate_table("simulations/results_cluster/sim_distributions/nrange10_30_100_130_200_230nsteps_1ngraphs_1/max_comcard_230step_compute_card_5/res_df.csv")

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &            &                     Accuracy &             Silhouette Score &               Davies-Bouldin &               Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                 &                              &                              \\
\midrule
Diversity Curves & Shortest Path &  \textbf{0.981 $\pm $ 0.021} &           0.290 $\pm $ 0.029 &           1.166 $\pm $ 0.079 &             82.271 $\pm $ 9.754 &           0.443 $\pm $ 0.139 &           0.255 $\pm $ 0.020 \\
                    & Shortest Path PCA &  \textbf{0.981 $\pm $ 0.021} &           0.301 $\pm $ 0.029 &  \textbf{1.132 $\pm $ 0.082} &            83.484 $\pm $ 10.040 &           

In [5]:
df

silh_mean    db_mean  \
Group               Method                                                   
Diversity Curves    Shortest Path                      0.289812   1.165834   
                    Shortest Path PCA                  0.300991   1.131798   
Diversity Profile   Profile Shortest Path              0.030053   4.619794   
Embeddings          NetLSD                            -0.210909   3.407263   
                    FEATHER                            0.041929   3.634665   
                    Graph2Vec                         -0.091115   7.520457   
                    Spectral Zoo                      -0.200770   9.383056   
Kernel              Kernel WL Optimal Assignment       0.012113        NaN   
                    Kernel Core Framework MDS          0.238698   2.482628   
                    Kernel Core Framework              0.226376        NaN   
                    Kernel WL MDS                      0.474476   1.976025   
                    Kernel Shortest Path               0.427666        NaN   
                    Kernel Shortest Path MDS           0.511629   1.456689   
                    Kernel WL Optimal Assignment MDS  -0.023919  11.148690   
                    Kernel WL                          0.397039        NaN   
                    Kernel Pyramid Match              -0.070033        NaN   
                    Kernel Pyramid Match MDS          -0.077564  38.517959   
                    Kernel Neighbourhood Hash MDS     -0.068374  40.118804   
                    Kernel Neighbourhood Hash         -0.056693        NaN   
TopER               TopER All Filtrations             -0.082212   3.495587   
                    TopER Popularity                  -0.123098   4.497648   
                    TopER Degree                      -0.117693   5.276161   
                    TopER Closeness                   -0.113239   5.711855   
                    TopER Olricci                     -0.132319  15.282224   
                    TopER Forricci                    -0.114760  14.682597   
Persistent Homology PH Degree                         -0.133355   1.922405   
                    PH Fricci                         -0.107084   1.739579   
                    PH Laplacian                      -0.141621   2.065011   
                    PH Olricci                        -0.111081   1.841083   

                                                         ch_mean  silh_std  \
Group               Method                                                   
Diversity Curves    Shortest Path                      82.271036  0.028577   
                    Shortest Path PCA                  83.484014  0.029466   
Diversity Profile   Profile Shortest Path              13.356664  0.020033   
Embeddings          NetLSD                              7.511256  0.055950   
                    FEATHER                            16.896370  0.016612   
                    Graph2Vec                           7.554280  0.023208   
                    Spectral Zoo                        5.844899  0.028457   
Kernel              Kernel WL Optimal Assignment             NaN  0.002031   
                    Kernel Core Framework MDS          45.601195  0.024136   
                    Kernel Core Framework                    NaN  0.021671   
                    Kernel WL MDS                      77.118715  0.041794   
                    Kernel Shortest Path                     NaN  0.045422   
                    Kernel Shortest Path MDS          136.742667  0.042603   
                    Kernel WL Optimal Assignment MDS    3.400037  0.009404   
                    Kernel WL                                NaN  0.037803   
                    Kernel Pyramid Match                     NaN  0.018400   
                    Kernel Pyramid Match MDS            0.513230  0.018575   
                    Kernel Neighbourhood Hash MDS       0.511862  0.014153   
                    Kernel Neighbourhood Hash                NaN  0.012912 

In [6]:
df = generate_table('simulations/results_cluster/sim_distributions/nrange10_30_100_130_200_230nsteps_1ngraphs_1/max_comcard_230step_compute_card_3/res_df.csv')

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &            &                     Accuracy &             Silhouette Score &               Davies-Bouldin &               Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                 &                              &                              \\
\midrule
Diversity Curves & Shortest Path &  \textbf{0.981 $\pm $ 0.021} &           0.293 $\pm $ 0.029 &           1.161 $\pm $ 0.079 &             83.044 $\pm $ 9.854 &           0.443 $\pm $ 0.139 &           0.255 $\pm $ 0.021 \\
                    & Shortest Path PCA &  \textbf{0.981 $\pm $ 0.021} &           0.304 $\pm $ 0.030 &  \textbf{1.127 $\pm $ 0.082} &            84.290 $\pm $ 10.148 &           

In [7]:
df

silh_mean    db_mean  \
Group               Method                                                   
Diversity Curves    Shortest Path                      0.292573   1.161217   
                    Shortest Path PCA                  0.304009   1.126866   
Diversity Profile   Profile Shortest Path              0.030053   4.619794   
Embeddings          NetLSD                            -0.210909   3.407263   
                    FEATHER                            0.041929   3.634665   
                    Graph2Vec                         -0.087310   7.927455   
                    Spectral Zoo                      -0.200770   9.383056   
Kernel              Kernel WL Optimal Assignment       0.012113        NaN   
                    Kernel Core Framework MDS          0.238698   2.482628   
                    Kernel Core Framework              0.226376        NaN   
                    Kernel WL MDS                      0.474476   1.976025   
                    Kernel Shortest Path               0.427666        NaN   
                    Kernel Shortest Path MDS           0.511629   1.456689   
                    Kernel WL Optimal Assignment MDS  -0.023919  11.148690   
                    Kernel WL                          0.397039        NaN   
                    Kernel Pyramid Match              -0.070033        NaN   
                    Kernel Pyramid Match MDS          -0.077564  38.517959   
                    Kernel Neighbourhood Hash MDS     -0.068374  40.118804   
                    Kernel Neighbourhood Hash         -0.056693        NaN   
TopER               TopER All Filtrations             -0.082212   3.495587   
                    TopER Popularity                  -0.123098   4.497648   
                    TopER Degree                      -0.117693   5.276161   
                    TopER Closeness                   -0.113239   5.711855   
                    TopER Olricci                     -0.132319  15.282224   
                    TopER Forricci                    -0.114760  14.682597   
Persistent Homology PH Degree                         -0.133355   1.922405   
                    PH Fricci                         -0.107084   1.739579   
                    PH Laplacian                      -0.141621   2.065011   
                    PH Olricci                        -0.111081   1.841083   

                                                         ch_mean  silh_std  \
Group               Method                                                   
Diversity Curves    Shortest Path                      83.044312  0.028672   
                    Shortest Path PCA                  84.289549  0.029582   
Diversity Profile   Profile Shortest Path              13.356664  0.020033   
Embeddings          NetLSD                              7.511256  0.055950   
                    FEATHER                            16.896370  0.016612   
                    Graph2Vec                           7.546876  0.024090   
                    Spectral Zoo                        5.844899  0.028457   
Kernel              Kernel WL Optimal Assignment             NaN  0.002031   
                    Kernel Core Framework MDS          45.601195  0.024136   
                    Kernel Core Framework                    NaN  0.021671   
                    Kernel WL MDS                      77.118715  0.041794   
                    Kernel Shortest Path                     NaN  0.045422   
                    Kernel Shortest Path MDS          136.742667  0.042603   
                    Kernel WL Optimal Assignment MDS    3.400037  0.009404   
                    Kernel WL                                NaN  0.037803   
                    Kernel Pyramid Match                     NaN  0.018400   
                    Kernel Pyramid Match MDS            0.513230  0.018575   
                    Kernel Neighbourhood Hash MDS       0.511862  0.014153   
                    Kernel Neighbourhood Hash                NaN  0.012912 

# exploring cluster results

In [9]:
import re
from pathlib import Path
import pandas as pd

ROOT = Path('simulations/results_cluster/sim_distributions')
DIVCURVES_METHODS = ['divcurves_shortest_path_distance', 'divcurves_pca_shortest_path_distance']

leaf_re = re.compile(r'max_comcard_(\d+)step_compute_card_(\d+)')
top_re = re.compile(r'nrange(?P<nrange>[\d_]+?)nsteps_(?P<nsteps>\d+)ngraphs_(?P<ngraphs>\d+)(?P<test>_test)?$')

rows = []
for csv_path in sorted(ROOT.glob('*/*/res_df.csv')):
    leaf_m = leaf_re.match(csv_path.parent.name)
    top_m = top_re.match(csv_path.parent.parent.name)
    max_comcard, step_compute_card = leaf_m.groups()
    nrange, nsteps, ngraphs, is_test = top_m.group('nrange', 'nsteps', 'ngraphs', 'test')

    res_df = pd.read_csv(csv_path, header=0, index_col=0)
    for method in DIVCURVES_METHODS:
        if method not in res_df.index:
            continue
        row = res_df.loc[method]
        rows.append({
            'top_config': csv_path.parent.parent.name,
            'leaf_config': csv_path.parent.name,
            'nrange': nrange, 'nsteps': int(nsteps), 'ngraphs': int(ngraphs), 'is_test': bool(is_test),
            'max_compute_card': int(max_comcard), 'step_compute_card': int(step_compute_card),
            'method': method,
            'acc_mean': row['acc_mean'], 'silh_mean': row['silh_mean'],
        })

divcurves_configs = pd.DataFrame(rows)
divcurves_configs['acc_silh_product'] = divcurves_configs['acc_mean'] * divcurves_configs['silh_mean']

for method, sub in divcurves_configs.groupby('method'):
    print(f'\n=== {method} ===')
    display(sub.sort_values('acc_silh_product', ascending=False)
               [['top_config', 'leaf_config', 'acc_mean', 'silh_mean', 'acc_silh_product']])



=== divcurves_pca_shortest_path_distance ===


,top_config,leaf_config,acc_mean,silh_mean,acc_silh_product
35,nrange10_40nsteps_1ngraphs_1_test,max_comcard_100step_compute_card_5,0.925000,0.401957,0.371810
9,nrange10_200nsteps_5ngraphs_1_test,max_comcard_200step_compute_card_2,0.967500,0.370578,0.358534
13,nrange10_200nsteps_5ngraphs_1_test,max_comcard_200step_compute_card_4,0.967500,0.369035,0.357041
11,nrange10_200nsteps_5ngraphs_1_test,max_comcard_200step_compute_card_3,0.967500,0.368171,0.356205
33,nrange10_30_100_130_200_230nsteps_1ngraphs_2,max_comcard_230step_compute_card_4,0.976562,0.321209,0.313680
31,nrange10_30_100_130_200_230nsteps_1ngraphs_2,max_comcard_230step_compute_card_2,0.975000,0.319714,0.311721
23,nrange10_30_100_130_200_230nsteps_1ngraphs_1,max_comcard_230step_compute_card_3,0.981250,0.304009,0.298309
25,nrange10_30_100_130_200_230nsteps_1ngraphs_1,max_comcard_230step_compute_card_5,0.981250,0.300991,0.295347
3,nrange10_200nsteps_10ngraphs_2,max_comcard_150step_compute_card_5,0.925000,0.287745,0.266164
7,nrange10_200nsteps_5ngraphs_1,max_comcard_150step_compute_card_5,0.954167,0.271984,0.259518



=== divcurves_shortest_path_distance ===


,top_config,leaf_config,acc_mean,silh_mean,acc_silh_product
34,nrange10_40nsteps_1ngraphs_1_test,max_comcard_100step_compute_card_5,0.925000,0.392253,0.362834
8,nrange10_200nsteps_5ngraphs_1_test,max_comcard_200step_compute_card_2,0.967500,0.356687,0.345095
12,nrange10_200nsteps_5ngraphs_1_test,max_comcard_200step_compute_card_4,0.967500,0.355203,0.343659
10,nrange10_200nsteps_5ngraphs_1_test,max_comcard_200step_compute_card_3,0.967500,0.354368,0.342851
32,nrange10_30_100_130_200_230nsteps_1ngraphs_2,max_comcard_230step_compute_card_4,0.968750,0.309900,0.300216
30,nrange10_30_100_130_200_230nsteps_1ngraphs_2,max_comcard_230step_compute_card_2,0.968750,0.308535,0.298893
22,nrange10_30_100_130_200_230nsteps_1ngraphs_1,max_comcard_230step_compute_card_3,0.981250,0.292573,0.287087
24,nrange10_30_100_130_200_230nsteps_1ngraphs_1,max_comcard_230step_compute_card_5,0.981250,0.289812,0.284378
2,nrange10_200nsteps_10ngraphs_2,max_comcard_150step_compute_card_5,0.918750,0.278256,0.255648
6,nrange10_200nsteps_5ngraphs_1,max_comcard_150step_compute_card_5,0.960833,0.263179,0.252871


In [10]:
df = generate_table("simulations/results_cluster/sim_distributions/nrange10_200nsteps_5ngraphs_1_test/max_comcard_200step_compute_card_2/res_df.csv")

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &              &                     Accuracy &             Silhouette Score &               Davies-Bouldin &              Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                &                              &                              \\
\midrule
Diversity Curves & Shortest Path &  \textbf{0.968 $\pm $ 0.044} &           0.357 $\pm $ 0.043 &           1.043 $\pm $ 0.154 &           60.706 $\pm $ 10.861 &           0.475 $\pm $ 0.129 &           0.241 $\pm $ 0.035 \\
                    & Shortest Path PCA &  \textbf{0.968 $\pm $ 0.044} &           0.371 $\pm $ 0.045 &  \textbf{1.008 $\pm $ 0.154} &           62.111 $\pm $ 11.252 &           0.

In [11]:
df

silh_mean    db_mean  \
Group               Method                                                   
Diversity Curves    Shortest Path                      0.356687   1.043369   
                    Shortest Path PCA                  0.370578   1.008308   
Diversity Profile   Profile Shortest Path             -0.007681   4.406536   
Embeddings          NetLSD                            -0.147845   2.927892   
                    FEATHER                           -0.006370   3.724330   
                    Graph2Vec                         -0.100741   5.041740   
                    Spectral Zoo                      -0.153843   6.370146   
Kernel              Kernel Shortest Path               0.493663        NaN   
                    Kernel Shortest Path MDS           0.538877   1.197897   
                    Kernel WL                          0.449668        NaN   
                    Kernel WL MDS                      0.506936   1.608595   
                    Kernel Core Framework MDS          0.233213   2.002970   
                    Kernel Core Framework              0.223384        NaN   
                    Kernel WL Optimal Assignment      -0.002962        NaN   
                    Kernel WL Optimal Assignment MDS  -0.035523   8.558962   
                    Kernel Pyramid Match              -0.098511        NaN   
                    Kernel Neighbourhood Hash         -0.076432        NaN   
                    Kernel Neighbourhood Hash MDS     -0.098984  22.695147   
                    Kernel Pyramid Match MDS          -0.108718  24.144552   
TopER               TopER All Filtrations             -0.119748   4.425378   
                    TopER Olricci                     -0.141758  24.143467   
                    TopER Popularity                  -0.152042   8.020626   
                    TopER Forricci                    -0.122474  22.785002   
                    TopER Closeness                   -0.153304  13.554608   
                    TopER Degree                      -0.156294   7.843699   
Persistent Homology PH Fricci                         -0.146607   1.852747   
                    PH Olricci                        -0.151973   1.867296   
                    PH Degree                         -0.219997   2.844157   
                    PH Laplacian                      -0.221977   3.121575   

                                                        ch_mean  silh_std  \
Group               Method                                                  
Diversity Curves    Shortest Path                     60.706088  0.043409   
                    Shortest Path PCA                 62.110907  0.045147   
Diversity Profile   Profile Shortest Path              7.411696  0.029897   
Embeddings          NetLSD                             3.819574  0.078724   
                    FEATHER                            9.354938  0.025027   
                    Graph2Vec                          9.888322  0.033467   
                    Spectral Zoo                       3.609380  0.039700   
Kernel              Kernel Shortest Path                    NaN  0.067069   
                    Kernel Shortest Path MDS          86.531319  0.063260   
                    Kernel WL                               NaN  0.057429   
                    Kernel WL MDS                     48.573952  0.062012   
                    Kernel Core Framework MDS         32.786450  0.039475   
                    Kernel Core Framework                   NaN  0.034666   
                    Kernel WL Optimal Assignment            NaN  0.002587   
                    Kernel WL Optimal Assignment MDS   3.214550  0.022109   
                    Kernel Pyramid Match                    NaN  0.017062   
                    Kernel Neighbourhood Hash               NaN  0.012392   
                    Kernel Neighbourhood Hash MDS      0.523809  0.013950   
                    Kernel Pyramid Match MDS           0.517605  0.017851   
TopER             

In [15]:
import re
from pathlib import Path
import pandas as pd

ROOT = Path('simulations/results_cluster/sim_distributions')
DIVCURVES_METHODS = ['divcurves_shortest_path_distance', 'divcurves_pca_shortest_path_distance']

METRIC = 'db_mean'  # <-- choose the clustering measure: 'silh_mean', 'db_mean', 'ch_mean', 'ari_agglo_mean', 'ari_kmeans_mean'
ASCENDING = {  # lower-is-better only for Davies-Bouldin
    'acc_mean': False, 'silh_mean': False, 'db_mean': True,
    'ch_mean': False, 'ari_agglo_mean': False, 'ari_kmeans_mean': False,
}

leaf_re = re.compile(r'max_comcard_(\d+)step_compute_card_(\d+)')
top_re = re.compile(r'nrange(?P<nrange>[\d_]+?)nsteps_(?P<nsteps>\d+)ngraphs_(?P<ngraphs>\d+)(?P<test>_test)?$')

rows = []
for csv_path in sorted(ROOT.glob('*/*/res_df.csv')):
    leaf_m = leaf_re.match(csv_path.parent.name)
    top_m = top_re.match(csv_path.parent.parent.name)
    max_comcard, step_compute_card = leaf_m.groups()
    nrange, nsteps, ngraphs, is_test = top_m.group('nrange', 'nsteps', 'ngraphs', 'test')

    res_df = pd.read_csv(csv_path, header=0, index_col=0)
    n_methods = len(res_df)
    res_df['acc_rank'] = res_df['acc_mean'].rank(ascending=ASCENDING['acc_mean'], method='min')
    res_df[f'{METRIC}_rank'] = res_df[METRIC].rank(ascending=ASCENDING[METRIC], method='min')

    for method in DIVCURVES_METHODS:
        if method not in res_df.index:
            continue
        row = res_df.loc[method]
        rows.append({
            'top_config': csv_path.parent.parent.name, 'leaf_config': csv_path.parent.name,
            'nrange': nrange, 'nsteps': int(nsteps), 'ngraphs': int(ngraphs), 'is_test': bool(is_test),
            'max_compute_card': int(max_comcard), 'step_compute_card': int(step_compute_card),
            'method': method, 'n_methods': n_methods,
            'acc_mean': row['acc_mean'], 'acc_rank': int(row['acc_rank']),
            METRIC: row[METRIC], f'{METRIC}_rank': int(row[f'{METRIC}_rank']),
        })

divcurves_configs = pd.DataFrame(rows)
divcurves_configs['rank_sum'] = divcurves_configs['acc_rank'] + divcurves_configs[f'{METRIC}_rank']

for method, sub in divcurves_configs.groupby('method'):
    print(f'\n=== {method} (ranked among all methods, metric={METRIC}) ===')
    display(sub.sort_values('rank_sum')
               [['top_config', 'leaf_config', 'n_methods', 'acc_mean', 'acc_rank', METRIC, f'{METRIC}_rank', 'rank_sum']])



=== divcurves_pca_shortest_path_distance (ranked among all methods, metric=db_mean) ===


,top_config,leaf_config,n_methods,acc_mean,acc_rank,db_mean,db_mean_rank,rank_sum
35,nrange10_40nsteps_1ngraphs_1_test,max_comcard_100step_compute_card_5,31,0.925000,1,0.796317,1,2
31,nrange10_30_100_130_200_230nsteps_1ngraphs_2,max_comcard_230step_compute_card_2,14,0.975000,1,1.062204,1,2
25,nrange10_30_100_130_200_230nsteps_1ngraphs_1,max_comcard_230step_compute_card_5,29,0.981250,1,1.131798,1,2
9,nrange10_200nsteps_5ngraphs_1_test,max_comcard_200step_compute_card_2,31,0.967500,1,1.008308,1,2
11,nrange10_200nsteps_5ngraphs_1_test,max_comcard_200step_compute_card_3,31,0.967500,1,1.011850,1,2
13,nrange10_200nsteps_5ngraphs_1_test,max_comcard_200step_compute_card_4,31,0.967500,1,1.010637,1,2
23,nrange10_30_100_130_200_230nsteps_1ngraphs_1,max_comcard_230step_compute_card_3,29,0.981250,1,1.126866,1,2
33,nrange10_30_100_130_200_230nsteps_1ngraphs_2,max_comcard_230step_compute_card_4,21,0.976562,1,1.059902,1,2
21,nrange10_30_100_130_200_230nsteps_1ngraphs_1,max_comcard_150step_compute_card_5,29,0.981250,2,1.436566,1,3
7,nrange10_200nsteps_5ngraphs_1,max_comcard_150step_compute_card_5,31,0.954167,3,1.168676,1,4



=== divcurves_shortest_path_distance (ranked among all methods, metric=db_mean) ===


,top_config,leaf_config,n_methods,acc_mean,acc_rank,db_mean,db_mean_rank,rank_sum
34,nrange10_40nsteps_1ngraphs_1_test,max_comcard_100step_compute_card_5,31,0.925000,1,0.812765,2,3
24,nrange10_30_100_130_200_230nsteps_1ngraphs_1,max_comcard_230step_compute_card_5,29,0.981250,1,1.165834,2,3
22,nrange10_30_100_130_200_230nsteps_1ngraphs_1,max_comcard_230step_compute_card_3,29,0.981250,1,1.161217,2,3
8,nrange10_200nsteps_5ngraphs_1_test,max_comcard_200step_compute_card_2,31,0.967500,1,1.043369,2,3
10,nrange10_200nsteps_5ngraphs_1_test,max_comcard_200step_compute_card_3,31,0.967500,1,1.047090,2,3
12,nrange10_200nsteps_5ngraphs_1_test,max_comcard_200step_compute_card_4,31,0.967500,1,1.045806,2,3
20,nrange10_30_100_130_200_230nsteps_1ngraphs_1,max_comcard_150step_compute_card_5,29,0.984375,1,1.449894,2,3
30,nrange10_30_100_130_200_230nsteps_1ngraphs_2,max_comcard_230step_compute_card_2,14,0.968750,2,1.102328,2,4
32,nrange10_30_100_130_200_230nsteps_1ngraphs_2,max_comcard_230step_compute_card_4,21,0.968750,2,1.100163,2,4
16,nrange10_30_100_130_200_230_300_330nsteps_1ngr...,max_comcard_150step_compute_card_5,14,0.981818,1,1.720221,4,5


In [16]:
df_cont = generate_table('simulations/results_cluster/sim_distributions/nrange10_200nsteps_5ngraphs_1_test/max_comcard_200step_compute_card_2/res_df.csv')

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &              &                     Accuracy &             Silhouette Score &               Davies-Bouldin &              Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                &                              &                              \\
\midrule
Diversity Curves & Shortest Path &  \textbf{0.968 $\pm $ 0.044} &           0.357 $\pm $ 0.043 &           1.043 $\pm $ 0.154 &           60.706 $\pm $ 10.861 &           0.475 $\pm $ 0.129 &           0.241 $\pm $ 0.035 \\
                    & Shortest Path PCA &  \textbf{0.968 $\pm $ 0.044} &           0.371 $\pm $ 0.045 &  \textbf{1.008 $\pm $ 0.154} &           62.111 $\pm $ 11.252 &           0.

In [17]:
df_cont

silh_mean    db_mean  \
Group               Method                                                   
Diversity Curves    Shortest Path                      0.356687   1.043369   
                    Shortest Path PCA                  0.370578   1.008308   
Diversity Profile   Profile Shortest Path             -0.007681   4.406536   
Embeddings          NetLSD                            -0.147845   2.927892   
                    FEATHER                           -0.006370   3.724330   
                    Graph2Vec                         -0.100741   5.041740   
                    Spectral Zoo                      -0.153843   6.370146   
Kernel              Kernel Shortest Path               0.493663        NaN   
                    Kernel Shortest Path MDS           0.538877   1.197897   
                    Kernel WL                          0.449668        NaN   
                    Kernel WL MDS                      0.506936   1.608595   
                    Kernel Core Framework MDS          0.233213   2.002970   
                    Kernel Core Framework              0.223384        NaN   
                    Kernel WL Optimal Assignment      -0.002962        NaN   
                    Kernel WL Optimal Assignment MDS  -0.035523   8.558962   
                    Kernel Pyramid Match              -0.098511        NaN   
                    Kernel Neighbourhood Hash         -0.076432        NaN   
                    Kernel Neighbourhood Hash MDS     -0.098984  22.695147   
                    Kernel Pyramid Match MDS          -0.108718  24.144552   
TopER               TopER All Filtrations             -0.119748   4.425378   
                    TopER Olricci                     -0.141758  24.143467   
                    TopER Popularity                  -0.152042   8.020626   
                    TopER Forricci                    -0.122474  22.785002   
                    TopER Closeness                   -0.153304  13.554608   
                    TopER Degree                      -0.156294   7.843699   
Persistent Homology PH Fricci                         -0.146607   1.852747   
                    PH Olricci                        -0.151973   1.867296   
                    PH Degree                         -0.219997   2.844157   
                    PH Laplacian                      -0.221977   3.121575   

                                                        ch_mean  silh_std  \
Group               Method                                                  
Diversity Curves    Shortest Path                     60.706088  0.043409   
                    Shortest Path PCA                 62.110907  0.045147   
Diversity Profile   Profile Shortest Path              7.411696  0.029897   
Embeddings          NetLSD                             3.819574  0.078724   
                    FEATHER                            9.354938  0.025027   
                    Graph2Vec                          9.888322  0.033467   
                    Spectral Zoo                       3.609380  0.039700   
Kernel              Kernel Shortest Path                    NaN  0.067069   
                    Kernel Shortest Path MDS          86.531319  0.063260   
                    Kernel WL                               NaN  0.057429   
                    Kernel WL MDS                     48.573952  0.062012   
                    Kernel Core Framework MDS         32.786450  0.039475   
                    Kernel Core Framework                   NaN  0.034666   
                    Kernel WL Optimal Assignment            NaN  0.002587   
                    Kernel WL Optimal Assignment MDS   3.214550  0.022109   
                    Kernel Pyramid Match                    NaN  0.017062   
                    Kernel Neighbourhood Hash               NaN  0.012392   
                    Kernel Neighbourhood Hash MDS      0.523809  0.013950   
                    Kernel Pyramid Match MDS           0.517605  0.017851   
TopER             

In [18]:
df_leaps = generate_table('simulations/results_cluster/sim_distributions/nrange10_30_100_130_200_230nsteps_1ngraphs_1/max_comcard_230step_compute_card_3/res_df.csv')

(([0.75], None), ([[0.9, 0.1]], [0.3333333333333333, 0.3333333333333333]), ([0.8, 0.05], [[0.25, 0.25, 0.25]]), ([0.25],))
\begin{tabular}{llllllll}
\toprule
                    &            &                     Accuracy &             Silhouette Score &               Davies-Bouldin &               Calinski-Harabasz &                  ARI k-means &            ARI agglomerative \\
{} & Method &                              &                              &                              &                                 &                              &                              \\
\midrule
Diversity Curves & Shortest Path &  \textbf{0.981 $\pm $ 0.021} &           0.293 $\pm $ 0.029 &           1.161 $\pm $ 0.079 &             83.044 $\pm $ 9.854 &           0.443 $\pm $ 0.139 &           0.255 $\pm $ 0.021 \\
                    & Shortest Path PCA &  \textbf{0.981 $\pm $ 0.021} &           0.304 $\pm $ 0.030 &  \textbf{1.127 $\pm $ 0.082} &            84.290 $\pm $ 10.148 &           

In [19]:
df_leaps

silh_mean    db_mean  \
Group               Method                                                   
Diversity Curves    Shortest Path                      0.292573   1.161217   
                    Shortest Path PCA                  0.304009   1.126866   
Diversity Profile   Profile Shortest Path              0.030053   4.619794   
Embeddings          NetLSD                            -0.210909   3.407263   
                    FEATHER                            0.041929   3.634665   
                    Graph2Vec                         -0.087310   7.927455   
                    Spectral Zoo                      -0.200770   9.383056   
Kernel              Kernel WL Optimal Assignment       0.012113        NaN   
                    Kernel Core Framework MDS          0.238698   2.482628   
                    Kernel Core Framework              0.226376        NaN   
                    Kernel WL MDS                      0.474476   1.976025   
                    Kernel Shortest Path               0.427666        NaN   
                    Kernel Shortest Path MDS           0.511629   1.456689   
                    Kernel WL Optimal Assignment MDS  -0.023919  11.148690   
                    Kernel WL                          0.397039        NaN   
                    Kernel Pyramid Match              -0.070033        NaN   
                    Kernel Pyramid Match MDS          -0.077564  38.517959   
                    Kernel Neighbourhood Hash MDS     -0.068374  40.118804   
                    Kernel Neighbourhood Hash         -0.056693        NaN   
TopER               TopER All Filtrations             -0.082212   3.495587   
                    TopER Popularity                  -0.123098   4.497648   
                    TopER Degree                      -0.117693   5.276161   
                    TopER Closeness                   -0.113239   5.711855   
                    TopER Olricci                     -0.132319  15.282224   
                    TopER Forricci                    -0.114760  14.682597   
Persistent Homology PH Degree                         -0.133355   1.922405   
                    PH Fricci                         -0.107084   1.739579   
                    PH Laplacian                      -0.141621   2.065011   
                    PH Olricci                        -0.111081   1.841083   

                                                         ch_mean  silh_std  \
Group               Method                                                   
Diversity Curves    Shortest Path                      83.044312  0.028672   
                    Shortest Path PCA                  84.289549  0.029582   
Diversity Profile   Profile Shortest Path              13.356664  0.020033   
Embeddings          NetLSD                              7.511256  0.055950   
                    FEATHER                            16.896370  0.016612   
                    Graph2Vec                           7.546876  0.024090   
                    Spectral Zoo                        5.844899  0.028457   
Kernel              Kernel WL Optimal Assignment             NaN  0.002031   
                    Kernel Core Framework MDS          45.601195  0.024136   
                    Kernel Core Framework                    NaN  0.021671   
                    Kernel WL MDS                      77.118715  0.041794   
                    Kernel Shortest Path                     NaN  0.045422   
                    Kernel Shortest Path MDS          136.742667  0.042603   
                    Kernel WL Optimal Assignment MDS    3.400037  0.009404   
                    Kernel WL                                NaN  0.037803   
                    Kernel Pyramid Match                     NaN  0.018400   
                    Kernel Pyramid Match MDS            0.513230  0.018575   
                    Kernel Neighbourhood Hash MDS       0.511862  0.014153   
                    Kernel Neighbourhood Hash                NaN  0.012912 